In [2]:
%pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import platform
import shutil
import subprocess


def ensure_ffmpeg() -> None:
    """Install ffmpeg on macOS / Linux (Colab uses apt; local Mac often uses conda or Homebrew)."""
    if shutil.which("ffmpeg"):
        print("ffmpeg: already on PATH")
        return

    sysname = platform.system()
    conda = shutil.which("conda")
    if conda:
        print("Installing ffmpeg via conda-forge (may take a minute)...")
        r = subprocess.run(
            [conda, "install", "-y", "-c", "conda-forge", "ffmpeg"],
            check=False,
        )
        if r.returncode == 0 and shutil.which("ffmpeg"):
            print("ffmpeg: installed with conda")
            return
        print("conda install did not expose ffmpeg; trying other options...")

    if sysname == "Darwin":
        brew = shutil.which("brew")
        if brew:
            print("Installing ffmpeg via Homebrew (may take a minute)...")
            r = subprocess.run([brew, "install", "ffmpeg"], check=False)
            if r.returncode == 0 and shutil.which("ffmpeg"):
                print("ffmpeg: installed with brew")
                return
        print(
            "Could not install ffmpeg automatically. In a terminal run one of:\n"
            "  conda install -y -c conda-forge ffmpeg\n"
            "  brew install ffmpeg"
        )
    elif sysname == "Linux":
        print("Installing ffmpeg via apt (needs sudo)...")
        r = subprocess.run(
            ["sudo", "apt-get", "install", "-y", "-q", "ffmpeg"],
            check=False,
        )
        if r.returncode == 0 and shutil.which("ffmpeg"):
            print("ffmpeg: installed with apt")
            return
        print("Try manually: sudo apt-get install -y ffmpeg")
    else:
        print(f"Install ffmpeg manually for {sysname}.")


ensure_ffmpeg()

%pip install -q \
    librosa==0.10.2 \
    soundfile==0.12.1 \
    audioread==3.0.1 \
    mir_eval==0.7 \
    music21==9.1.0 \
    pretty_midi==0.2.10 \
    scikit-learn==1.4.2 \
    numpy==1.26.4 \
    scipy==1.13.0 \
    pandas==2.2.2 \
    matplotlib==3.9.0 \
    tqdm==4.66.4 \
    musdb


In [4]:
# utils.py

import numpy as np

# =========================================================
# HYPERPARAMETERS
# =========================================================

EPSILON = 1e-8
MAX_CIRCLE_STEPS = 6.0
MINOR_TO_MAJOR_OFFSET = 3
DTW_FALLBACK_DIAGONAL = True
COSINE_TO_SIMILARITY = True


# =========================================================
# CONSTANTS
# =========================================================

PITCH_TO_PC = {
    "C": 0, "C#": 1, "Db": 1, "D": 2, "D#": 3, "Eb": 3, "E": 4, "Fb": 4, "E#": 5, "F": 5,
    "F#": 6, "Gb": 6, "G": 7, "G#": 8, "Ab": 8, "A": 9, "A#": 10, "Bb": 10,
    "B": 11, "Cb": 11, "B#": 0
}

PC_TO_NAME = ["C", "C#", "D", "Eb", "E", "F", "F#", "G", "Ab", "A", "Bb", "B"]

CIRCLE_OF_FIFTHS_INDEX = {
    0: 0,   # C
    7: 1,   # G
    2: 2,   # D
    9: 3,   # A
    4: 4,   # E
    11: 5,  # B
    6: 6,   # F#
    1: 7,   # C#
    8: 8,   # Ab
    3: 9,   # Eb
    10: 10, # Bb
    5: 11,  # F
}


# =========================================================
# FUNCTIONS
# =========================================================

def _relative_major_pc(pc: int, scale: str) -> int:
    if scale.lower().startswith("min"):
        return (pc + MINOR_TO_MAJOR_OFFSET) % 12
    return pc % 12


def circle_of_fifths_distance(key1: str, scale1: str, key2: str, scale2: str):
    pc1 = PITCH_TO_PC.get(key1)
    pc2 = PITCH_TO_PC.get(key2)

    if pc1 is None or pc2 is None:
        return None, None

    r1 = _relative_major_pc(pc1, scale1)
    r2 = _relative_major_pc(pc2, scale2)

    i1 = CIRCLE_OF_FIFTHS_INDEX[r1]
    i2 = CIRCLE_OF_FIFTHS_INDEX[r2]

    steps = abs(i1 - i2)
    steps = min(steps, 12 - steps)

    dist_norm = steps / MAX_CIRCLE_STEPS

    return steps, dist_norm


def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    a = np.asarray(a, dtype=float).reshape(-1)
    b = np.asarray(b, dtype=float).reshape(-1)

    if a.size == 0 or b.size == 0 or a.size != b.size:
        return 0.0

    na = np.linalg.norm(a) + EPSILON
    nb = np.linalg.norm(b) + EPSILON

    return float(np.dot(a, b) / (na * nb))


def dtw_cosine(X: np.ndarray, Y: np.ndarray) -> float:
    from scipy.spatial.distance import cdist

    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)

    if X.ndim == 1:
        X = X[:, None]
    if Y.ndim == 1:
        Y = Y[:, None]

    if len(X) == 0 or len(Y) == 0:
        return 0.0

    Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + EPSILON)
    Yn = Y / (np.linalg.norm(Y, axis=1, keepdims=True) + EPSILON)

    C = cdist(Xn, Yn, metric="cosine")
    C = np.nan_to_num(C, nan=1.0, posinf=1.0, neginf=1.0)

    try:
        import librosa
        _, wp = librosa.sequence.dtw(C=C)
        path = wp[::-1]
    except Exception:
        if DTW_FALLBACK_DIAGONAL:
            t = min(len(Xn), len(Yn))
            path = [(i, i) for i in range(t)]
        else:
            return 0.0

    if COSINE_TO_SIMILARITY:
        sims = [1.0 - C[i, j] for (i, j) in path]
    else:
        sims = [C[i, j] for (i, j) in path]

    return float(np.mean(sims)) if sims else 0.0

In [5]:
# structural_form.py

from typing import Dict, Any, Optional, Tuple
import numpy as np
import mir_eval
from scipy.signal import find_peaks
from sklearn.cluster import KMeans
import librosa

# =========================================================
# HYPERPARAMETERS
# =========================================================

SAMPLE_RATE = 22050
N_MFCC = 13
HOP_LENGTH = 512

NOVELTY_SMOOTH_WINDOW = 16
PEAK_MIN_DISTANCE_SEC = 10.0
PEAK_PROMINENCE = 0.1

FALLBACK_SEGMENT_SEC = 10.0
MIN_SEGMENTS_FALLBACK = 3

BOUNDARY_WINDOW_SEC = 0.5
BOUNDARY_SHIFT_SEARCH_RANGE_SEC = 2.0
BOUNDARY_SHIFT_STEP_SEC = 0.02

MIN_INTERVAL_DURATION = 1e-10

MAX_KMEANS_CLUSTERS = 8
MIN_KMEANS_CLUSTERS = 2
KMEANS_RANDOM_STATE = 0
KMEANS_N_INIT = 10

STRUCTURE_BOUNDARY_WEIGHT = 0.7
STRUCTURE_ARI_WEIGHT = 0.3

DEFAULT_FALLBACK_DURATION = 30.0

# =========================================================
# INTERNAL HELPERS
# =========================================================

def _bounds_to_intervals(bounds) -> np.ndarray:
    bounds = np.asarray(bounds, dtype=float)

    if bounds.ndim == 1:
        bounds = bounds[np.isfinite(bounds)]
        bounds = np.unique(bounds)
        if bounds.size < 2:
            return np.empty((0, 2), dtype=float)
        return np.column_stack([bounds[:-1], bounds[1:]])

    if bounds.size == 0:
        return np.empty((0, 2), dtype=float)

    bounds = bounds.astype(float, copy=False)
    if bounds.shape[1] != 2:
        return np.empty((0, 2), dtype=float)

    keep = np.isfinite(bounds).all(axis=1) & ((bounds[:, 1] - bounds[:, 0]) > MIN_INTERVAL_DURATION)
    return bounds[keep]


def _shift_intervals(intervals: np.ndarray, delta: float) -> np.ndarray:
    intervals = np.asarray(intervals, dtype=float)
    if intervals.size == 0:
        return intervals
    return intervals + np.array([delta, delta], dtype=float)


def _crop_overlap_for_detection(
    ref_intervals: np.ndarray,
    est_intervals: np.ndarray,
) -> Tuple[Optional[np.ndarray], Optional[np.ndarray]]:
    ref_intervals = np.asarray(ref_intervals, dtype=float)
    est_intervals = np.asarray(est_intervals, dtype=float)

    if ref_intervals.size == 0 or est_intervals.size == 0:
        return None, None

    t_min = max(0.0, float(ref_intervals[0, 0]), float(est_intervals[0, 0]))
    t_max = min(float(ref_intervals[-1, 1]), float(est_intervals[-1, 1]))

    if not (t_max > t_min):
        return None, None

    def _clip_valid(intervals: np.ndarray) -> Optional[np.ndarray]:
        if intervals.size == 0:
            return None

        starts = np.maximum(intervals[:, 0], t_min)
        ends = np.minimum(intervals[:, 1], t_max)
        keep = (ends - starts) > MIN_INTERVAL_DURATION

        if not np.any(keep):
            return None

        clipped = np.column_stack([starts[keep], ends[keep]])

        # normalize to start at 0 for mir_eval stability
        clipped = clipped - np.array([t_min, t_min], dtype=float)
        return clipped

    ref_clipped = _clip_valid(ref_intervals)
    est_clipped = _clip_valid(est_intervals)

    if ref_clipped is None or est_clipped is None:
        return None, None

    return ref_clipped, est_clipped


def _merge_short_segments(
    bounds: np.ndarray,
    labels: list,
    min_duration: float = 2.0,
) -> Tuple[np.ndarray, list]:
    bounds = np.asarray(bounds, dtype=float)
    if bounds.ndim != 1 or bounds.size < 2:
        return bounds, labels

    intervals = _bounds_to_intervals(bounds)
    if len(intervals) == 0 or len(labels) != len(intervals):
        return bounds, labels

    new_bounds = [float(bounds[0])]
    new_labels = []

    cur_start = float(bounds[0])
    cur_end = float(bounds[1])
    cur_label = labels[0]

    for i in range(1, len(labels)):
        seg_start = float(bounds[i])
        seg_end = float(bounds[i + 1])
        seg_label = labels[i]

        cur_duration = cur_end - cur_start

        if cur_duration < min_duration or seg_label == cur_label:
            cur_end = seg_end
        else:
            new_bounds.append(cur_end)
            new_labels.append(cur_label)
            cur_start = seg_start
            cur_end = seg_end
            cur_label = seg_label

    new_bounds.append(cur_end)
    new_labels.append(cur_label)

    new_bounds = np.asarray(new_bounds, dtype=float)
    return new_bounds, new_labels


# =========================================================
# PUBLIC HELPERS
# =========================================================

def best_boundary_shift(
    ref_bounds,
    est_bounds,
    window: float = BOUNDARY_WINDOW_SEC,
    search_range: float = BOUNDARY_SHIFT_SEARCH_RANGE_SEC,
    step: float = BOUNDARY_SHIFT_STEP_SEC,
):
    ref_intervals = _bounds_to_intervals(ref_bounds)
    est_intervals = _bounds_to_intervals(est_bounds)

    best = {"F": -1.0, "delta": 0.0, "P": 0.0, "R": 0.0}

    if ref_intervals.size == 0 or est_intervals.size == 0:
        return best

    for delta in np.arange(-search_range, search_range + 1e-12, step):
        est_shifted = _shift_intervals(est_intervals, delta)
        ref_crop, est_crop = _crop_overlap_for_detection(ref_intervals, est_shifted)

        if ref_crop is None or est_crop is None:
            continue

        try:
            precision, recall, f_measure = mir_eval.segment.detection(
                ref_crop, est_crop, window=window, trim=True
            )
        except Exception:
            continue

        if f_measure > best["F"]:
            best.update({
                "F": float(f_measure),
                "delta": float(delta),
                "P": float(precision),
                "R": float(recall),
            })

    if best["F"] < 0.0:
        best = {"F": 0.0, "delta": 0.0, "P": 0.0, "R": 0.0}

    return best


def bounds_to_intervals(bounds: np.ndarray) -> np.ndarray:
    return _bounds_to_intervals(bounds)


def crop_to_overlap(intervals, labels, t_min, t_max):
    intervals = np.asarray(intervals, dtype=float)

    if intervals.size == 0:
        return intervals, ([] if labels is not None else None)

    starts = np.maximum(intervals[:, 0], t_min)
    ends = np.minimum(intervals[:, 1], t_max)
    keep = (ends - starts) > MIN_INTERVAL_DURATION

    intervals_cropped = np.column_stack([starts[keep], ends[keep]])

    if intervals_cropped.size == 0:
        return np.empty((0, 2), dtype=float), ([] if labels is not None else None)

    # normalize to start at 0 for mir_eval.segment.pairwise / ari
    intervals_cropped = intervals_cropped - np.array([t_min, t_min], dtype=float)

    labels_cropped = None
    if labels is not None:
        labels_array = np.asarray(labels, dtype=object)
        labels_cropped = labels_array[keep].tolist()

        if len(labels_cropped) < len(intervals_cropped):
            filler = labels_cropped[-1] if labels_cropped else "S"
            labels_cropped += [filler] * (len(intervals_cropped) - len(labels_cropped))
        elif len(labels_cropped) > len(intervals_cropped):
            labels_cropped = labels_cropped[:len(intervals_cropped)]

    return intervals_cropped, labels_cropped


def mir_segment_metrics(ref_bounds, est_bounds, ref_labels=None, est_labels=None):
    ref_intervals = bounds_to_intervals(ref_bounds)
    est_intervals = bounds_to_intervals(est_bounds)

    if ref_intervals.size == 0 or est_intervals.size == 0:
        return {"boundary_p": 0.0, "boundary_r": 0.0, "boundary_f": 0.0}

    t_min = max(float(ref_intervals[0, 0]), float(est_intervals[0, 0]), 0.0)
    t_max = min(float(ref_intervals[-1, 1]), float(est_intervals[-1, 1]))

    if not (t_max > t_min):
        return {"boundary_p": 0.0, "boundary_r": 0.0, "boundary_f": 0.0}

    ref_cropped, ref_labels_cropped = crop_to_overlap(ref_intervals, ref_labels, t_min, t_max)
    est_cropped, est_labels_cropped = crop_to_overlap(est_intervals, est_labels, t_min, t_max)

    if ref_cropped.size == 0 or est_cropped.size == 0:
        return {"boundary_p": 0.0, "boundary_r": 0.0, "boundary_f": 0.0}

    precision, recall, f_measure = mir_eval.segment.detection(
        ref_cropped,
        est_cropped,
        window=BOUNDARY_WINDOW_SEC,
        trim=True,
    )

    out = {
        "boundary_p": float(precision),
        "boundary_r": float(recall),
        "boundary_f": float(f_measure),
    }

    labels_ok = (
        ref_labels_cropped is not None
        and est_labels_cropped is not None
        and len(ref_labels_cropped) == len(ref_cropped)
        and len(est_labels_cropped) == len(est_cropped)
    )

    if labels_ok:
        try:
            p_pair, r_pair, f_pair = mir_eval.segment.pairwise(
                ref_cropped, ref_labels_cropped, est_cropped, est_labels_cropped
            )
            out.update({
                "pairwise_p": float(p_pair),
                "pairwise_r": float(r_pair),
                "pairwise_f": float(f_pair),
            })
        except Exception as exc:
            out["pairwise_error"] = str(exc)

        try:
            ari = mir_eval.segment.ari(
                ref_cropped, ref_labels_cropped, est_cropped, est_labels_cropped
            )
            out["ari"] = float(ari)
        except Exception as exc:
            out["ari_error"] = str(exc)

    return out


# =========================================================
# SEGMENT EXTRACTION
# =========================================================

def librosa_segments(audio_path: str):
    try:
        y, sr = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)
        duration = float(len(y)) / float(sr)

        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC, hop_length=HOP_LENGTH)

        recurrence = librosa.segment.recurrence_matrix(
            mfcc, mode="affinity", sym=True
        )
        novelty = librosa.segment.recurrence_to_lag(recurrence, pad=True)
        novelty = np.max(novelty, axis=0)

        smooth_window = np.hanning(NOVELTY_SMOOTH_WINDOW)
        if smooth_window.sum() > 0:
            smooth_window = smooth_window / smooth_window.sum()
        novelty = np.convolve(novelty, smooth_window, mode="same")

        frame_duration = HOP_LENGTH / sr
        min_peak_distance = max(1, int(PEAK_MIN_DISTANCE_SEC / frame_duration))

        peaks, _ = find_peaks(
            novelty,
            distance=min_peak_distance,
            prominence=PEAK_PROMINENCE,
        )

        bound_times = np.r_[0.0, peaks * frame_duration, duration]
        bound_times = np.clip(bound_times, 0.0, duration)
        bound_times = np.unique(bound_times)

        intervals = _bounds_to_intervals(bound_times)
        n_segments = len(intervals)

        if n_segments < 2:
            bound_times = np.linspace(
                0.0,
                duration,
                max(MIN_SEGMENTS_FALLBACK, int(duration / FALLBACK_SEGMENT_SEC) + 1),
            )
            intervals = _bounds_to_intervals(bound_times)
            n_segments = len(intervals)

        seg_features = []
        for start, end in intervals:
            start_frame = int(start / frame_duration)
            end_frame = max(start_frame + 1, int(end / frame_duration))
            end_frame = min(end_frame, mfcc.shape[1])

            segment_slice = mfcc[:, start_frame:end_frame]
            if segment_slice.size == 0:
                seg_features.append(np.zeros((N_MFCC,), dtype=float))
            else:
                seg_features.append(np.mean(segment_slice, axis=1))

        seg_features = np.asarray(seg_features, dtype=float)

        n_clusters = min(
            MAX_KMEANS_CLUSTERS,
            max(MIN_KMEANS_CLUSTERS, n_segments // 2),
        )

        if len(seg_features) < n_clusters:
            n_clusters = max(1, len(seg_features))

        if n_clusters <= 1:
            labels = ["S0"] * max(1, n_segments)
            return bound_times, labels

        kmeans = KMeans(
            n_clusters=n_clusters,
            random_state=KMEANS_RANDOM_STATE,
            n_init=KMEANS_N_INIT,
        ).fit(seg_features)

        labels = [f"S{cluster}" for cluster in kmeans.labels_]

        bound_times, labels = _merge_short_segments(bound_times, labels, min_duration=2.0)
        return bound_times, labels

    except Exception:
        try:
            y, sr = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)
            duration = float(len(y)) / float(sr)
        except Exception:
            duration = DEFAULT_FALLBACK_DURATION

        n_segments = max(MIN_KMEANS_CLUSTERS, int(duration / FALLBACK_SEGMENT_SEC))
        bounds = np.linspace(0.0, duration, n_segments + 1)
        labels = [f"S{i}" for i in range(n_segments)]
        return bounds, labels


# =========================================================
# MAIN API
# =========================================================

def structural_score(audio_ref: str, audio_est: str) -> Dict[str, Any]:
    ref_bounds, ref_labels = librosa_segments(audio_ref)
    est_bounds, est_labels = librosa_segments(audio_est)

    best = best_boundary_shift(
        ref_bounds,
        est_bounds,
        window=BOUNDARY_WINDOW_SEC,
        search_range=BOUNDARY_SHIFT_SEARCH_RANGE_SEC,
        step=BOUNDARY_SHIFT_STEP_SEC,
    )

    est_bounds_shifted = np.asarray(est_bounds, dtype=float) + best["delta"]
    scores = mir_segment_metrics(ref_bounds, est_bounds_shifted, ref_labels, est_labels)

    boundary_f_after_shift = max(0.0, safe_float(best["F"]) if "safe_float" in globals() else float(best["F"]))
    ari = float(scores.get("ari", 0.0)) if np.isfinite(scores.get("ari", 0.0)) else 0.0

    scores.update({
        "best_boundary_shift_sec": float(best["delta"]),
        "boundary_f_after_shift": float(boundary_f_after_shift),
        "structural_form_score": float(
            STRUCTURE_BOUNDARY_WEIGHT * boundary_f_after_shift
            + STRUCTURE_ARI_WEIGHT * ari
        ),
    })

    return scores

In [6]:
# harmony_tonality.py

from typing import Dict, Any, Tuple, List
import numpy as np
import librosa
import mir_eval

# =========================================================
# HYPERPARAMETERS
# =========================================================

SAMPLE_RATE = 22050
HOP_LENGTH = 512

CHROMA_METHOD = "cqt"          # "cqt", "stft", "cens"
CHROMA_NORM = np.inf
CHROMA_SMOOTH_FRAMES = 9

KEY_PROFILE_MAJOR = np.array(
    [6.35, 2.23, 3.48, 2.33, 4.38, 4.09, 2.52, 5.19, 2.39, 3.66, 2.29, 2.88],
    dtype=float,
)
KEY_PROFILE_MINOR = np.array(
    [6.33, 2.68, 3.52, 5.38, 2.60, 3.53, 2.54, 4.75, 3.98, 2.69, 3.34, 3.17],
    dtype=float,
)

PITCH_NAMES = ["C", "C#", "D", "Eb", "E", "F", "F#", "G", "Ab", "A", "Bb", "B"]

CHORD_TEMPLATE_MAJOR = np.array([1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0], dtype=float)
CHORD_TEMPLATE_MINOR = np.array([1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0], dtype=float)

CHORD_FRAME_SMOOTH = 5
MIN_VALID_DURATION_SEC = 1e-6


# =========================================================
# INTERNAL HELPERS
# =========================================================

def _safe_float(x, default=np.nan):
    try:
        x = float(x)
        if np.isnan(x) or np.isinf(x):
            return default
        return x
    except Exception:
        return default


def _load_audio(audio_path: str, sr: int = SAMPLE_RATE):
    y, sr = librosa.load(audio_path, sr=sr, mono=True)
    y = np.nan_to_num(y, nan=0.0)
    return y, sr


def _moving_average_1d(x: np.ndarray, win: int) -> np.ndarray:
    if win <= 1:
        return x
    kernel = np.ones(win, dtype=float) / float(win)
    return np.convolve(x, kernel, mode="same")


def _smooth_chroma(chroma: np.ndarray, win: int = CHROMA_SMOOTH_FRAMES) -> np.ndarray:
    if chroma.ndim != 2 or chroma.shape[0] != 12:
        return chroma
    out = np.empty_like(chroma, dtype=float)
    for i in range(12):
        out[i] = _moving_average_1d(chroma[i], win)
    return out


def _framewise_normalize(X: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    X = np.asarray(X, dtype=float)
    if X.ndim != 2:
        return X
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    return X / (norms + eps)


def _extract_chroma(audio_path: str, sr: int = SAMPLE_RATE, method: str = CHROMA_METHOD) -> np.ndarray:
    y, sr = _load_audio(audio_path, sr=sr)

    if method == "stft":
        chroma = librosa.feature.chroma_stft(
            y=y,
            sr=sr,
            hop_length=HOP_LENGTH,
            norm=CHROMA_NORM,
        )
    elif method == "cens":
        chroma = librosa.feature.chroma_cens(
            y=y,
            sr=sr,
            hop_length=HOP_LENGTH,
        )
    else:
        chroma = librosa.feature.chroma_cqt(
            y=y,
            sr=sr,
            hop_length=HOP_LENGTH,
            norm=CHROMA_NORM,
        )

    chroma = np.asarray(chroma, dtype=float)
    chroma = _smooth_chroma(chroma, win=CHROMA_SMOOTH_FRAMES)
    return chroma


def _best_transposition_cosine(mean_ref: np.ndarray, mean_est: np.ndarray) -> Tuple[float, int]:
    best_score = -1.0
    best_shift = 0

    for shift in range(12):
        rolled = np.roll(mean_est, shift)
        score = cosine_sim(mean_ref, rolled)
        if score > best_score:
            best_score = score
            best_shift = shift

    return float(best_score), int(best_shift)


def _best_transposition_dtw(C0: np.ndarray, C1: np.ndarray) -> Tuple[float, int]:
    best_score = -1.0
    best_shift = 0

    for shift in range(12):
        rolled = np.roll(C1, shift, axis=1)
        score = dtw_cosine(C0, rolled)
        if score > best_score:
            best_score = score
            best_shift = shift

    return float(best_score), int(best_shift)


def _major_minor_key_from_chroma(mean_chroma: np.ndarray) -> Dict[str, Any]:
    best_key = None
    best_scale = None
    best_corr = -np.inf

    mean_chroma = np.asarray(mean_chroma, dtype=float).reshape(-1)
    if mean_chroma.size != 12 or not np.isfinite(mean_chroma).any():
        return {"error": "invalid chroma"}

    for i in range(12):
        for profile, scale_name in (
            (KEY_PROFILE_MAJOR, "major"),
            (KEY_PROFILE_MINOR, "minor"),
        ):
            rotated = np.roll(profile, i)
            corr = np.corrcoef(mean_chroma, rotated)[0, 1]
            if np.isfinite(corr) and corr > best_corr:
                best_corr = corr
                best_key = PITCH_NAMES[i]
                best_scale = scale_name

    return {
        "key": best_key,
        "scale": best_scale,
        "strength": _safe_float(best_corr, 0.0),
    }


def _build_chord_templates() -> Dict[str, np.ndarray]:
    templates = {}
    for i, name in enumerate(PITCH_NAMES):
        templates[f"{name}:maj"] = np.roll(CHORD_TEMPLATE_MAJOR, i)
        templates[f"{name}:min"] = np.roll(CHORD_TEMPLATE_MINOR, i)
    return templates


def _smooth_labels(labels: List[str], win: int = CHORD_FRAME_SMOOTH) -> List[str]:
    if win <= 1 or len(labels) == 0:
        return labels

    half = win // 2
    out = []
    for i in range(len(labels)):
        lo = max(0, i - half)
        hi = min(len(labels), i + half + 1)
        chunk = labels[lo:hi]
        vals, counts = np.unique(chunk, return_counts=True)
        out.append(vals[np.argmax(counts)])
    return out


# =========================================================
# KEY
# =========================================================

def key_scale_music21(audio_path: str) -> Dict[str, Any]:
    try:
        chroma = _extract_chroma(audio_path, sr=SAMPLE_RATE, method=CHROMA_METHOD)
        mean_chroma = chroma.mean(axis=1)
        return _major_minor_key_from_chroma(mean_chroma)
    except Exception as exc:
        return {"error": f"key extraction failed: {exc}"}


def key_relatedness(ref_key: str, ref_scale: str, est_key: str, est_scale: str) -> Dict[str, Any]:
    steps, norm = circle_of_fifths_distance(ref_key, ref_scale, est_key, est_scale)
    if steps is None:
        return {"distance_steps": None, "distance_norm_0to1": None}
    return {
        "distance_steps": int(steps),
        "distance_norm_0to1": float(norm),
    }


# =========================================================
# CHROMA
# =========================================================

def chroma_similarity(
    audio_ref: str,
    audio_est: str,
    sr: int = SAMPLE_RATE,
    method: str = CHROMA_METHOD,
) -> Dict[str, float]:
    C0 = _extract_chroma(audio_ref, sr=sr, method=method).T   # [frames, 12]
    C1 = _extract_chroma(audio_est, sr=sr, method=method).T   # [frames, 12]

    if len(C0) == 0 or len(C1) == 0:
        return {
            "chroma_dtw_cosine": 0.0,
            "mean_chroma_cosine": 0.0,
            "best_shift_chroma_dtw_cosine": 0.0,
            "best_shift_mean_chroma_cosine": 0.0,
            "best_shift_steps": 0.0,
        }

    C0n = _framewise_normalize(C0)
    C1n = _framewise_normalize(C1)

    mean0 = C0n.mean(axis=0)
    mean1 = C1n.mean(axis=0)

    raw_dtw = dtw_cosine(C0n, C1n)
    raw_mean = cosine_sim(mean0, mean1)

    best_mean, best_mean_shift = _best_transposition_cosine(mean0, mean1)
    best_dtw, best_dtw_shift = _best_transposition_dtw(C0n, C1n)

    # usually these should agree, but keep the DTW-derived shift as the main one
    return {
        "chroma_dtw_cosine": _safe_float(raw_dtw, 0.0),
        "mean_chroma_cosine": _safe_float(raw_mean, 0.0),
        "best_shift_chroma_dtw_cosine": _safe_float(best_dtw, 0.0),
        "best_shift_mean_chroma_cosine": _safe_float(best_mean, 0.0),
        "best_shift_steps": float(best_dtw_shift),
    }


# =========================================================
# CHORDS
# =========================================================

def chord_sequences_librosa(audio_path: str):
    try:
        chroma = _extract_chroma(audio_path, sr=SAMPLE_RATE, method=CHROMA_METHOD)
        templates = _build_chord_templates()

        frame_duration = HOP_LENGTH / SAMPLE_RATE
        labels = []

        for frame in chroma.T:
            best_label = "N"
            best_score = -np.inf

            for chord_name, template in templates.items():
                score = float(np.dot(frame, template))
                if score > best_score:
                    best_score = score
                    best_label = chord_name

            labels.append(best_label)

        labels = _smooth_labels(labels, win=CHORD_FRAME_SMOOTH)

        if len(labels) == 0:
            return None, None

        intervals = []
        out_labels = []

        start = 0.0
        current = labels[0]

        for i, lab in enumerate(labels[1:], start=1):
            if lab != current:
                end = i * frame_duration
                if end - start > MIN_VALID_DURATION_SEC:
                    intervals.append([start, end])
                    out_labels.append(current)
                start = end
                current = lab

        final_end = len(labels) * frame_duration
        if final_end - start > MIN_VALID_DURATION_SEC:
            intervals.append([start, final_end])
            out_labels.append(current)

        if len(intervals) == 0:
            return None, None

        return np.asarray(intervals, dtype=float), out_labels

    except Exception:
        return None, None


def chord_similarity_mireval(audio_ref: str, audio_est: str) -> Dict[str, float]:
    try:
        iv_ref, lb_ref = chord_sequences_librosa(audio_ref)
        iv_est, lb_est = chord_sequences_librosa(audio_est)

        if iv_ref is None or iv_est is None or len(iv_ref) == 0 or len(iv_est) == 0:
            return {"error": "chord extraction failed"}

        scores = mir_eval.chord.evaluate(iv_ref, lb_ref, iv_est, lb_est)
        return {f"mir_{k}": float(v) for k, v in scores.items()}

    except Exception as exc:
        return {"error": f"chord evaluation failed: {exc}"}


# =========================================================
# MAIN
# =========================================================

def harmony_score(audio_ref: str, audio_est: str) -> Dict[str, Any]:
    result = {}

    key_ref = key_scale_music21(audio_ref)
    key_est = key_scale_music21(audio_est)

    result["key_ref"] = key_ref
    result["key_est"] = key_est

    if "key" in key_ref and "key" in key_est:
        result["key_relatedness"] = key_relatedness(
            key_ref["key"], key_ref["scale"],
            key_est["key"], key_est["scale"],
        )

    result["chroma_similarity"] = chroma_similarity(audio_ref, audio_est)
    result["chord_similarity"] = chord_similarity_mireval(audio_ref, audio_est)

    return result

In [7]:
# melody_motifs.py

from typing import Dict, Any, List
import numpy as np
import librosa
import mir_eval
from scipy.interpolate import interp1d
from scipy.spatial.distance import cdist

# =========================================================
# HYPERPARAMETERS
# =========================================================

SAMPLE_RATE = 16000
FMIN = 55.0
FMAX = 1760.0
HOP_LENGTH = 512
FRAME_LENGTH = 2048
MAX_DURATION = 60.0

COMMON_GRID_FPS = 25.0
CONTOUR_GRID_FPS = 25.0
MOTIF_GRID_FPS = 10.0

MEDIAN_FILTER_WIDTH = 5
MIN_COMMON_GRID_POINTS = 2
MIN_CONTOUR_VALID_POINTS = 3
MIN_CONTOUR_DIFF_POINTS = 2
MIN_VOICED_RUN = 4

SEMITONE_IN_CENTS = 100.0
REFERENCE_FREQUENCY_HZ = 440.0
EPSILON = 1e-8

MOTIF_NGRAM_N = 3
PITCH_SHIFT_SEMITONE_DIVISOR = 12.0

# =========================================================
# F0 EXTRACTION
# =========================================================

def f0_librosa(
    audio_path: str,
    sr: int = SAMPLE_RATE,
    fmin: float = FMIN,
    fmax: float = FMAX,
    hop_length: int = HOP_LENGTH,
    max_duration: float | None = MAX_DURATION,
):
    y, sr = librosa.load(audio_path, sr=sr, mono=True)

    if max_duration is not None:
        max_len = int(sr * max_duration)
        if len(y) > max_len:
            y = y[:max_len]

    y = np.nan_to_num(y, nan=0.0)

    if len(y) == 0 or np.max(np.abs(y)) < EPSILON:
        return np.array([], dtype=float), np.array([], dtype=float), np.array([], dtype=bool)

    try:
        f0, vflag, _ = librosa.pyin(
            y,
            fmin=fmin,
            fmax=fmax,
            sr=sr,
            hop_length=hop_length,
            frame_length=FRAME_LENGTH,
        )
        times = librosa.times_like(f0, sr=sr, hop_length=hop_length)
        return times, np.asarray(f0, dtype=float), np.asarray(vflag, dtype=bool)
    except Exception:
        pass

    try:
        f0 = librosa.yin(
            y,
            fmin=fmin,
            fmax=fmax,
            sr=sr,
            hop_length=hop_length,
            frame_length=FRAME_LENGTH,
        )
        times = librosa.times_like(f0, sr=sr, hop_length=hop_length)
        f0 = np.asarray(f0, dtype=float)
        vflag = np.isfinite(f0) & (f0 >= fmin) & (f0 <= fmax)
        f0 = f0.copy()
        f0[~vflag] = np.nan
        return times, f0, vflag
    except Exception:
        return np.array([], dtype=float), np.array([], dtype=float), np.array([], dtype=bool)

# =========================================================
# INTERNAL HELPERS
# =========================================================

def _safe_float(x, default=np.nan):
    try:
        x = float(x)
        if np.isnan(x) or np.isinf(x):
            return default
        return x
    except Exception:
        return default


def _hz_to_cents(f: np.ndarray, tonic: float = REFERENCE_FREQUENCY_HZ) -> np.ndarray:
    f = np.asarray(f, dtype=float)
    out = np.full_like(f, np.nan, dtype=float)
    valid = np.isfinite(f) & (f > 0)
    out[valid] = 1200.0 * np.log2(f[valid] / tonic)
    return out


def _interp_masked(times_src, values_src, times_dst, kind="linear"):
    times_src = np.asarray(times_src, dtype=float)
    values_src = np.asarray(values_src, dtype=float)
    times_dst = np.asarray(times_dst, dtype=float)

    valid = np.isfinite(times_src) & np.isfinite(values_src)
    if valid.sum() < MIN_COMMON_GRID_POINTS:
        return np.full_like(times_dst, np.nan, dtype=float)

    f = interp1d(
        times_src[valid],
        values_src[valid],
        kind=kind,
        bounds_error=False,
        fill_value=np.nan,
        assume_sorted=True,
    )
    return np.asarray(f(times_dst), dtype=float)


def _interp_bool(times_src, values_src, times_dst):
    times_src = np.asarray(times_src, dtype=float)
    values_src = np.asarray(values_src, dtype=bool)
    times_dst = np.asarray(times_dst, dtype=float)

    valid = np.isfinite(times_src)
    if valid.sum() < 1:
        return np.zeros_like(times_dst, dtype=bool)

    f = interp1d(
        times_src[valid],
        values_src[valid].astype(float),
        kind="nearest",
        bounds_error=False,
        fill_value=0.0,
        assume_sorted=True,
    )
    return np.asarray(f(times_dst) > 0.5, dtype=bool)


def _median_filter_pitch(f0: np.ndarray, width: int = MEDIAN_FILTER_WIDTH) -> np.ndarray:
    f0 = np.asarray(f0, dtype=float).copy()
    if len(f0) == 0:
        return f0

    out = f0.copy()
    half = width // 2

    for i in range(len(f0)):
        lo = max(0, i - half)
        hi = min(len(f0), i + half + 1)
        window = f0[lo:hi]
        window = window[np.isfinite(window) & (window > 0)]
        if window.size:
            out[i] = np.median(window)

    return out


def _prepare_common_grid(
    t_ref,
    f0_ref,
    v_ref,
    t_est,
    f0_est,
    v_est,
    fps: float = COMMON_GRID_FPS,
):
    t_ref = np.asarray(t_ref, dtype=float)
    f0_ref = np.asarray(f0_ref, dtype=float)
    v_ref = np.asarray(v_ref, dtype=bool)

    t_est = np.asarray(t_est, dtype=float)
    f0_est = np.asarray(f0_est, dtype=float)
    v_est = np.asarray(v_est, dtype=bool)

    if len(t_ref) == 0 or len(t_est) == 0:
        return None

    t0 = max(float(t_ref[0]), float(t_est[0]))
    t1 = min(float(t_ref[-1]), float(t_est[-1]))

    if not (t1 > t0 + 1e-6):
        return None

    n_points = max(MIN_COMMON_GRID_POINTS, int((t1 - t0) * fps) + 1)
    grid = np.linspace(t0, t1, n_points)

    f_ref = _interp_masked(t_ref, f0_ref, grid, kind="linear")
    f_est = _interp_masked(t_est, f0_est, grid, kind="linear")

    v_ref_grid = _interp_bool(t_ref, v_ref, grid)
    v_est_grid = _interp_bool(t_est, v_est, grid)

    f_ref[~v_ref_grid] = np.nan
    f_est[~v_est_grid] = np.nan

    f_ref = _median_filter_pitch(f_ref, width=MEDIAN_FILTER_WIDTH)
    f_est = _median_filter_pitch(f_est, width=MEDIAN_FILTER_WIDTH)

    return grid, f_ref, v_ref_grid, f_est, v_est_grid


def _dtw_cosine(X: np.ndarray, Y: np.ndarray) -> float:
    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)

    if X.ndim == 1:
        X = X[:, None]
    if Y.ndim == 1:
        Y = Y[:, None]

    if len(X) == 0 or len(Y) == 0:
        return 0.0

    D = cdist(X, Y, metric="cosine")
    D = np.nan_to_num(D, nan=1.0, posinf=1.0, neginf=1.0)

    n, m = D.shape
    acc = np.full((n + 1, m + 1), np.inf, dtype=float)
    acc[0, 0] = 0.0

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            acc[i, j] = D[i - 1, j - 1] + min(
                acc[i - 1, j],
                acc[i, j - 1],
                acc[i - 1, j - 1],
            )

    cost = acc[n, m] / max(n + m, 1)
    sim = 1.0 - cost
    return float(np.clip(sim, 0.0, 1.0))

# =========================================================
# MIR-EVAL MELODY METRICS
# =========================================================

def mir_melody_scores(t_ref, f0_ref, v_ref, t_est, f0_est, v_est) -> Dict[str, float]:
    prep = _prepare_common_grid(
        t_ref, f0_ref, v_ref,
        t_est, f0_est, v_est,
        fps=COMMON_GRID_FPS,
    )

    if prep is None:
        return {
            "overall_accuracy": 0.0,
            "raw_pitch_accuracy": 0.0,
            "raw_chroma_accuracy": 0.0,
            "voicing_recall": 0.0,
            "voicing_false_alarm": 0.0,
        }

    _, f_ref_hz, v_ref_grid, f_est_hz, v_est_grid = prep

    ref_cent = _hz_to_cents(f_ref_hz)
    est_cent = _hz_to_cents(f_est_hz)

    ref_cent_0 = np.where(v_ref_grid & np.isfinite(ref_cent), ref_cent, 0.0)
    est_cent_0 = np.where(v_est_grid & np.isfinite(est_cent), est_cent, 0.0)

    try:
        scores = {
            "overall_accuracy": mir_eval.melody.overall_accuracy(v_ref_grid, ref_cent_0, v_est_grid, est_cent_0),
            "raw_pitch_accuracy": mir_eval.melody.raw_pitch_accuracy(v_ref_grid, ref_cent_0, v_est_grid, est_cent_0),
            "raw_chroma_accuracy": mir_eval.melody.raw_chroma_accuracy(v_ref_grid, ref_cent_0, v_est_grid, est_cent_0),
            "voicing_recall": mir_eval.melody.voicing_recall(v_ref_grid, v_est_grid),
            "voicing_false_alarm": mir_eval.melody.voicing_false_alarm(v_ref_grid, v_est_grid),
        }
    except Exception:
        scores = {
            "overall_accuracy": 0.0,
            "raw_pitch_accuracy": 0.0,
            "raw_chroma_accuracy": 0.0,
            "voicing_recall": 0.0,
            "voicing_false_alarm": 0.0,
        }

    return {k: _safe_float(v, 0.0) for k, v in scores.items()}

# =========================================================
# CONTOUR
# =========================================================

def contour_dtw_distance(t_ref, f0_ref, t_est, f0_est) -> Dict[str, float]:
    t_ref = np.asarray(t_ref, dtype=float)
    f0_ref = np.asarray(f0_ref, dtype=float)
    t_est = np.asarray(t_est, dtype=float)
    f0_est = np.asarray(f0_est, dtype=float)

    if len(t_ref) < 2 or len(t_est) < 2:
        return {"pitch_dtw_cosine": 0.0}

    t0 = max(float(t_ref[0]), float(t_est[0]))
    t1 = min(float(t_ref[-1]), float(t_est[-1]))

    if not (t1 > t0 + 1e-6):
        return {"pitch_dtw_cosine": 0.0}

    grid = np.linspace(t0, t1, int((t1 - t0) * CONTOUR_GRID_FPS) + 1)

    f_ref = _interp_masked(t_ref, f0_ref, grid, kind="linear")
    f_est = _interp_masked(t_est, f0_est, grid, kind="linear")

    valid = np.isfinite(f_ref) & np.isfinite(f_est) & (f_ref > 0) & (f_est > 0)
    if valid.sum() < MIN_CONTOUR_VALID_POINTS:
        return {"pitch_dtw_cosine": 0.0}

    ref_cents = _hz_to_cents(f_ref[valid])
    est_cents = _hz_to_cents(f_est[valid])

    d_ref = np.diff(ref_cents)
    d_est = np.diff(est_cents)

    if len(d_ref) < MIN_CONTOUR_DIFF_POINTS or len(d_est) < MIN_CONTOUR_DIFF_POINTS:
        return {"pitch_dtw_cosine": 0.0}

    sim = _dtw_cosine(d_ref[:, None], d_est[:, None])
    return {"pitch_dtw_cosine": _safe_float(sim, 0.0)}

# =========================================================
# MOTIFS
# =========================================================

def interval_tokens(
    t: np.ndarray,
    f0: np.ndarray,
    v: np.ndarray = None,
    fps: float = MOTIF_GRID_FPS,
    min_voiced_run: int = MIN_VOICED_RUN,
) -> List[int]:
    t = np.asarray(t, dtype=float)
    f0 = np.asarray(f0, dtype=float)

    if v is None:
        v = np.isfinite(f0) & (f0 > 0)
    else:
        v = np.asarray(v, dtype=bool)

    if len(t) < 2:
        return []

    grid = np.linspace(t[0], t[-1], int((t[-1] - t[0]) * fps) + 1)
    f_grid = _interp_masked(t, f0, grid, kind="linear")
    v_grid = _interp_bool(t, v, grid)

    f_grid[~v_grid] = np.nan
    cents = _hz_to_cents(f_grid)

    valid = np.isfinite(cents)
    tokens: List[int] = []

    i = 0
    while i < len(cents):
        if not valid[i]:
            i += 1
            continue

        j = i
        while j < len(cents) and valid[j]:
            j += 1

        if (j - i) >= min_voiced_run:
            run = cents[i:j]
            steps = np.diff(run) / SEMITONE_IN_CENTS
            steps = np.round(steps).astype(int)
            tokens.extend(int(step) for step in steps if np.isfinite(step))

        i = j

    return tokens


def motif_ngram_overlap(
    t_ref,
    f0_ref,
    t_est,
    f0_est,
    v_ref=None,
    v_est=None,
    n: int = MOTIF_NGRAM_N,
) -> Dict[str, float]:
    tokens_ref = interval_tokens(t_ref, f0_ref, v_ref)
    tokens_est = interval_tokens(t_est, f0_est, v_est)

    def grams(xs):
        return {tuple(xs[i:i + n]) for i in range(len(xs) - n + 1)} if len(xs) >= n else set()

    grams_ref = grams(tokens_ref)
    grams_est = grams(tokens_est)

    if not grams_ref and not grams_est:
        return {"jaccard": 1.0, "recall": 1.0}

    if not grams_ref:
        return {"jaccard": 0.0, "recall": 0.0}

    inter = len(grams_ref & grams_est)
    union = len(grams_ref | grams_est)

    return {
        "jaccard": float(inter / (union + EPSILON)),
        "recall": float(inter / (len(grams_ref) + EPSILON)),
    }

# =========================================================
# MAIN
# =========================================================

def melody_score(audio_ref: str, audio_est: str, use_essentia: bool = False) -> Dict[str, Any]:
    t_ref, f0_ref, v_ref = f0_librosa(audio_ref)
    t_est, f0_est, v_est = f0_librosa(audio_est)

    if len(t_ref) == 0 or len(t_est) == 0:
        return {"error": "F0 extraction failed"}

    out = {}
    out["mir_melody"] = mir_melody_scores(t_ref, f0_ref, v_ref, t_est, f0_est, v_est)
    out["contour_dtw"] = contour_dtw_distance(t_ref, f0_ref, t_est, f0_est)
    out["motif_overlap_n3"] = motif_ngram_overlap(
        t_ref, f0_ref, t_est, f0_est, v_ref, v_est, n=MOTIF_NGRAM_N
    )
    return out


def melody_score_pitch_shift_aware(audio_ref: str, audio_est: str, n_steps: int = 6) -> Dict[str, Any]:
    ratio = 2 ** (n_steps / PITCH_SHIFT_SEMITONE_DIVISOR)

    t_ref, f0_ref, v_ref = f0_librosa(audio_ref)
    t_est, f0_est, v_est = f0_librosa(audio_est)

    if len(t_ref) == 0 or len(t_est) == 0:
        return {"error": "F0 extraction failed"}

    f0_est_norm = np.asarray(f0_est, dtype=float).copy()
    valid = np.isfinite(f0_est_norm) & (f0_est_norm > 0)
    f0_est_norm[valid] = f0_est_norm[valid] / ratio

    out = {}
    out["raw"] = {
        "mir_melody": mir_melody_scores(t_ref, f0_ref, v_ref, t_est, f0_est, v_est),
        "contour_dtw": contour_dtw_distance(t_ref, f0_ref, t_est, f0_est),
        "motif_overlap_n3": motif_ngram_overlap(
            t_ref, f0_ref, t_est, f0_est, v_ref, v_est, n=MOTIF_NGRAM_N
        ),
    }
    out["normalized"] = {
        "mir_melody": mir_melody_scores(t_ref, f0_ref, v_ref, t_est, f0_est_norm, v_est),
        "contour_dtw": contour_dtw_distance(t_ref, f0_ref, t_est, f0_est_norm),
        "motif_overlap_n3": motif_ngram_overlap(
            t_ref, f0_ref, t_est, f0_est_norm, v_ref, v_est, n=MOTIF_NGRAM_N
        ),
    }
    return out

In [8]:
# rhythm_meter.py

from typing import Dict, Any, Tuple
import numpy as np
import librosa
import mir_eval

# =========================================================
# HYPERPARAMETERS
# =========================================================

SAMPLE_RATE = 22050
BEAT_TRACK_UNITS = "time"
BEAT_TRACK_TRIM = True

DEFAULT_BEAT_PERIOD_SEC = 0.5
MIN_BEATS_FOR_PERIOD = 2

OFFSET_GOOD_FRACTION_OF_PERIOD = 0.6
OFFSET_GRID_HALF_WIDTH_IN_PERIODS = 0.5
OFFSET_GRID_STEPS = 41

BEAT_MATCH_TOLERANCE_SEC = 0.07
HALF_DOUBLE_RATIO = 2.0
EPSILON = 1e-8

# =========================================================
# INTERNAL HELPERS
# =========================================================

def _safe_float(x, default=np.nan):
    try:
        x = float(x)
        if np.isnan(x) or np.isinf(x):
            return default
        return x
    except Exception:
        return default


def _as_1d_float_array(x) -> np.ndarray:
    arr = np.asarray(x, dtype=float).reshape(-1)
    arr = arr[np.isfinite(arr)]
    return arr


def _normalize_beats(beats) -> np.ndarray:
    beats = _as_1d_float_array(beats)
    if beats.size == 0:
        return beats
    beats = np.unique(beats)
    return beats


def _folded_bpm_delta(bpm_ref: float, bpm_est: float) -> float:
    bpm_ref = _safe_float(bpm_ref, default=np.nan)
    bpm_est = _safe_float(bpm_est, default=np.nan)

    if not np.isfinite(bpm_ref) or not np.isfinite(bpm_est) or bpm_ref <= 0 or bpm_est <= 0:
        return np.nan

    candidates = [
        bpm_est,
        bpm_est * HALF_DOUBLE_RATIO,
        bpm_est / HALF_DOUBLE_RATIO,
    ]
    return float(min(abs(bpm_ref - c) for c in candidates if c > 0))


# =========================================================
# BEAT / TEMPO EXTRACTION
# =========================================================

def tempo_and_beats_librosa(audio_path: str, sr: int = SAMPLE_RATE) -> Tuple[float, np.ndarray]:
    y, sr = librosa.load(audio_path, sr=sr, mono=True)
    y = np.nan_to_num(y, nan=0.0)

    tempo, beats = librosa.beat.beat_track(
        y=y,
        sr=sr,
        units=BEAT_TRACK_UNITS,
        trim=BEAT_TRACK_TRIM,
    )

    if hasattr(tempo, "__len__") and len(np.atleast_1d(tempo)) > 0:
        tempo_val = float(np.atleast_1d(tempo)[0])
    else:
        tempo_val = float(tempo)

    beats = _normalize_beats(beats)
    return tempo_val, beats


# =========================================================
# MIR_EVAL BEAT METRICS
# =========================================================

def beat_fmeasure_mireval(ref_beats, est_beats) -> Dict[str, float]:
    ref_beats = _normalize_beats(ref_beats)
    est_beats = _normalize_beats(est_beats)

    if ref_beats.size == 0 or est_beats.size == 0:
        return {
            "F-measure": 0.0,
            "Cemgil": 0.0,
            "Cemgil Best Metric Level": 0.0,
            "Goto": 0.0,
            "P-score": 0.0,
            "Correct Metric Level Continuous": 0.0,
            "Correct Metric Level Total": 0.0,
            "Any Metric Level Continuous": 0.0,
            "Any Metric Level Total": 0.0,
            "Information gain": 0.0,
        }

    try:
        scores = mir_eval.beat.evaluate(ref_beats, est_beats)
        return {k: float(v) for k, v in scores.items()}
    except Exception:
        return {
            "F-measure": 0.0,
            "Cemgil": 0.0,
            "Cemgil Best Metric Level": 0.0,
            "Goto": 0.0,
            "P-score": 0.0,
            "Correct Metric Level Continuous": 0.0,
            "Correct Metric Level Total": 0.0,
            "Any Metric Level Continuous": 0.0,
            "Any Metric Level Total": 0.0,
            "Information gain": 0.0,
        }


# =========================================================
# BEAT PERIOD / OFFSET
# =========================================================

def _estimate_beat_period_from_all_beats(all_beats_times: np.ndarray) -> float:
    all_beats_times = _normalize_beats(all_beats_times)

    if all_beats_times.size < MIN_BEATS_FOR_PERIOD:
        return DEFAULT_BEAT_PERIOD_SEC

    d = np.diff(all_beats_times)
    d = d[np.isfinite(d) & (d > 0)]

    if d.size == 0:
        return DEFAULT_BEAT_PERIOD_SEC

    return float(np.median(d))


def _best_offset_grid(
    ref_times: np.ndarray,
    est_times: np.ndarray,
    beat_period: float,
    tol: float,
) -> Tuple[float, int]:
    ref_times = _normalize_beats(ref_times)
    est_times = _normalize_beats(est_times)

    if ref_times.size == 0 or est_times.size == 0:
        return 0.0, 0

    diffs = []
    for t in ref_times:
        j = int(np.argmin(np.abs(est_times - t)))
        diffs.append(est_times[j] - t)

    diffs = np.asarray(diffs, dtype=float)

    good = np.abs(diffs) <= OFFSET_GOOD_FRACTION_OF_PERIOD * beat_period
    delta0 = float(np.median(diffs[good])) if np.any(good) else 0.0

    grid = np.linspace(
        delta0 - OFFSET_GRID_HALF_WIDTH_IN_PERIODS * beat_period,
        delta0 + OFFSET_GRID_HALF_WIDTH_IN_PERIODS * beat_period,
        OFFSET_GRID_STEPS,
    )

    def match_cnt(delta: float) -> int:
        i = 0
        j = 0
        tp = 0

        est_shift = est_times + delta

        while i < len(ref_times) and j < len(est_shift):
            d = est_shift[j] - ref_times[i]

            if abs(d) <= tol:
                tp += 1
                i += 1
                j += 1
            elif d < -tol:
                j += 1
            else:
                i += 1

        return tp

    counts = np.array([match_cnt(delta) for delta in grid], dtype=int)
    best_idx = int(np.argmax(counts))

    return float(grid[best_idx]), int(counts[best_idx])


# =========================================================
# MAIN API
# =========================================================

def rhythm_score(audio_ref: str, audio_est: str) -> Dict[str, Any]:
    out: Dict[str, Any] = {}

    tempo_ref_bpm, beats_ref = tempo_and_beats_librosa(audio_ref)
    tempo_est_bpm, beats_est = tempo_and_beats_librosa(audio_est)

    beat_period_ref = _estimate_beat_period_from_all_beats(beats_ref)
    beat_period_est = _estimate_beat_period_from_all_beats(beats_est)
    beat_period = float(np.nanmedian([beat_period_ref, beat_period_est]))

    best_offset_sec, best_offset_matches = _best_offset_grid(
        beats_ref,
        beats_est,
        beat_period=beat_period,
        tol=BEAT_MATCH_TOLERANCE_SEC,
    )

    beats_est_aligned = beats_est + best_offset_sec
    beat_scores = beat_fmeasure_mireval(beats_ref, beats_est_aligned)

    out["tempo_ref_bpm"] = _safe_float(tempo_ref_bpm, 0.0)
    out["tempo_est_bpm"] = _safe_float(tempo_est_bpm, 0.0)

    # Raw BPM delta
    raw_delta = abs(out["tempo_ref_bpm"] - out["tempo_est_bpm"])
    out["delta_bpm"] = float(raw_delta)

    # Folded BPM delta (robust to half/double tempo)
    out["delta_bpm_folded"] = _safe_float(
        _folded_bpm_delta(out["tempo_ref_bpm"], out["tempo_est_bpm"]),
        0.0,
    )

    out["beat_period_sec"] = _safe_float(beat_period, DEFAULT_BEAT_PERIOD_SEC)
    out["best_offset_sec"] = _safe_float(best_offset_sec, 0.0)
    out["best_offset_matches"] = int(best_offset_matches)

    out["n_ref_beats"] = int(len(beats_ref))
    out["n_est_beats"] = int(len(beats_est))

    out["beat_mir_eval"] = beat_scores
    return out

In [9]:
import numpy as np

# fix deprecated numpy aliases
np.int = int
np.float = float
np.complex = complex
np.bool = bool
np.object = object
np.str = str

print("✓ numpy aliases patched")

✓ numpy aliases patched


In [10]:
import os
from pathlib import Path

# =========================================================
# CONFIG — WAV list for later cells (`downloaded`)
# =========================================================
# Rendered from Lakh MIDI via scripts/midi_samples_to_wav.py
AUDIO_WAV_DIR = os.path.join("music", "lmd_100_samples_wav")

_wav_dir = Path(AUDIO_WAV_DIR)
if not _wav_dir.is_dir():
    raise FileNotFoundError(
        f"Missing directory {AUDIO_WAV_DIR!r}. Create WAVs with: python scripts/midi_samples_to_wav.py"
    )

_wavs = sorted(_wav_dir.glob("*.wav"), key=lambda x: x.name)
if not _wavs:
    raise FileNotFoundError(
        f"No .wav files in {AUDIO_WAV_DIR!r}. Run: python scripts/midi_samples_to_wav.py"
    )

downloaded = [str(p.resolve()) for p in _wavs]

print(f"✓ Loaded {len(downloaded)} tracks from {AUDIO_WAV_DIR}")
print("Done.")


✓ Loaded 98 tracks from music/lmd_100_samples_wav
Done.


In [ ]:
# =========================================================
# PITCH-SHIFT EDIT + EVAL PIPELINE
# For WAV paths in `downloaded` (MUSDB exports or music/lmd_100_samples_wav/*.wav)
# =========================================================
#
# Assumes these are already defined/imported in your Colab:
#   - downloaded  (list of original wav paths)
#   - harmony_score
#   - rhythm_score
#   - structural_score
#   - melody_score_pitch_shift_aware
#
# Example imports:
# from harmony_tonality import harmony_score
# from rhythm_meter import rhythm_score
# from structural_form import structural_score
# from melody_motifs import melody_score_pitch_shift_aware
#
# =========================================================

import os
import json
import numpy as np
import librosa
import soundfile as sf
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# =========================================================
# HYPERPARAMETERS
# =========================================================

PITCH_SHIFT_STEPS = 6
AUDIO_EDIT_DIR = "audio_edited/pitch_shift"
RESULTS_JSON_PATH = "pitch_shift_results.json"

# I/O parallelism for generating edited audio
NUM_AUDIO_WORKERS = 8

# Eval batching / checkpointing
EVAL_BATCH_SIZE = 32
SAVE_EVERY_BATCH = True

# =========================================================
# SETUP
# =========================================================

os.makedirs(AUDIO_EDIT_DIR, exist_ok=True)

# =========================================================
# HELPERS
# =========================================================

def safe_float(x, default=np.nan):
    try:
        x = float(x)
        if np.isnan(x) or np.isinf(x):
            return default
        return x
        צע
    except Exception:
        return default


def valid_vals(xs):
    out = []
    for x in xs:
        v = safe_float(x)
        if np.isfinite(v):
            out.append(v)
    return out


def summarize(vals):
    vals = valid_vals(vals)
    if not vals:
        return np.nan, np.nan, 0
    return float(np.median(vals)), float(np.std(vals)), len(vals)


def save_json(path, obj):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)


def make_pitch_shifted_path(src_path: str, out_dir: str = AUDIO_EDIT_DIR) -> str:
    base = os.path.basename(src_path)
    return os.path.join(out_dir, base)


# =========================================================
# 1) CREATE PITCH-SHIFTED COPIES
# =========================================================

def create_pitch_shifted_file(src_path: str, n_steps: int = PITCH_SHIFT_STEPS) -> str:
    out_path = make_pitch_shifted_path(src_path)

    if os.path.exists(out_path):
        return out_path

    y, sr = librosa.load(src_path, sr=None, mono=True)
    y_shifted = librosa.effects.pitch_shift(y, sr=sr, n_steps=n_steps)
    sf.write(out_path, y_shifted, sr)
    return out_path


pitch_shifted = [None] * len(downloaded)

with ThreadPoolExecutor(max_workers=NUM_AUDIO_WORKERS) as executor:
    future_to_idx = {
        executor.submit(create_pitch_shifted_file, src_path, PITCH_SHIFT_STEPS): idx
        for idx, src_path in enumerate(downloaded)
    }

    for future in tqdm(as_completed(future_to_idx), total=len(future_to_idx), desc="Creating pitch-shifted audio"):
        idx = future_to_idx[future]
        try:
            pitch_shifted[idx] = future.result()
        except Exception as e:
            print(f"⚠️ Failed pitch shift for {downloaded[idx]}: {e}")
            pitch_shifted[idx] = None

pairs = [(ref, est) for ref, est in zip(downloaded, pitch_shifted) if est is not None]

print(f"✓ Created {len(pairs)} valid pitch-shifted pairs")

# =========================================================
# 2) EVALUATE
# =========================================================

def evaluate_pair(ref: str, est: str, n_steps: int = PITCH_SHIFT_STEPS) -> dict:
    return {
        "ref": ref,
        "est": est,
        "edit_type": "pitch_shift",
        "n_steps": n_steps,
        "harmony_tonality": harmony_score(ref, est),
        "rhythm_meter": rhythm_score(ref, est),
        "structural_form": structural_score(ref, est),
        "melodic_content": melody_score_pitch_shift_aware(ref, est, n_steps=n_steps),
    }


all_results = []

for batch_start in tqdm(range(0, len(pairs), EVAL_BATCH_SIZE), desc="Evaluating batches"):
    batch = pairs[batch_start:batch_start + EVAL_BATCH_SIZE]

    batch_results = []
    for ref, est in batch:
        try:
            batch_results.append(evaluate_pair(ref, est, PITCH_SHIFT_STEPS))
        except Exception as e:
            batch_results.append({
                "ref": ref,
                "est": est,
                "edit_type": "pitch_shift",
                "n_steps": PITCH_SHIFT_STEPS,
                "error": str(e),
            })

    all_results.extend(batch_results)

    if SAVE_EVERY_BATCH:
        save_json(RESULTS_JSON_PATH, all_results)

    print(f"✓ batch {batch_start // EVAL_BATCH_SIZE + 1} done ({min(batch_start + EVAL_BATCH_SIZE, len(pairs))}/{len(pairs)})")

save_json(RESULTS_JSON_PATH, all_results)
print(f"✓ Saved raw results to {RESULTS_JSON_PATH}")

# =========================================================
# 3) EXTRACT METRICS
# =========================================================

def extract_metrics(results):
    metrics = {
        # harmony
        "harmony_key_distance": [],
        "harmony_chroma_dtw": [],
        "harmony_chroma_mean": [],
        "harmony_best_shift_chroma_dtw": [],
        "harmony_best_shift_chroma_mean": [],
        "harmony_chord_root": [],
        "harmony_chord_majmin": [],

        # rhythm
        "rhythm_delta_bpm": [],
        "rhythm_delta_bpm_folded": [],
        "rhythm_beat_fmeasure": [],
        "rhythm_info_gain": [],

        # structure
        "structure_boundary_f": [],
        "structure_ari": [],
        "structure_score": [],

        # melody raw
        "melody_raw_overall_acc": [],
        "melody_raw_voicing_recall": [],
        "melody_raw_pitch_acc": [],
        "melody_raw_chroma_acc": [],
        "melody_raw_contour_dtw": [],
        "melody_raw_motif_jaccard": [],
        "melody_raw_motif_recall": [],

        # melody normalized
        "melody_norm_overall_acc": [],
        "melody_norm_voicing_recall": [],
        "melody_norm_pitch_acc": [],
        "melody_norm_chroma_acc": [],
        "melody_norm_contour_dtw": [],
        "melody_norm_motif_jaccard": [],
        "melody_norm_motif_recall": [],
    }

    for r in results:
        if "error" in r:
            continue

        h = r.get("harmony_tonality", {})
        rm = r.get("rhythm_meter", {})
        s = r.get("structural_form", {})
        m = r.get("melodic_content", {})

        # ---------------- harmony ----------------
        metrics["harmony_key_distance"].append(
            safe_float(h.get("key_relatedness", {}).get("distance_norm_0to1", np.nan))
        )
        metrics["harmony_chroma_dtw"].append(
            safe_float(h.get("chroma_similarity", {}).get("chroma_dtw_cosine", np.nan))
        )
        metrics["harmony_chroma_mean"].append(
            safe_float(h.get("chroma_similarity", {}).get("mean_chroma_cosine", np.nan))
        )
        metrics["harmony_best_shift_chroma_dtw"].append(
            safe_float(h.get("chroma_similarity", {}).get("best_shift_chroma_dtw_cosine", np.nan))
        )
        metrics["harmony_best_shift_chroma_mean"].append(
            safe_float(h.get("chroma_similarity", {}).get("best_shift_mean_chroma_cosine", np.nan))
        )
        metrics["harmony_chord_root"].append(
            safe_float(h.get("chord_similarity", {}).get("mir_root", np.nan))
        )
        metrics["harmony_chord_majmin"].append(
            safe_float(h.get("chord_similarity", {}).get("mir_majmin", np.nan))
        )

        # ---------------- rhythm ----------------
        metrics["rhythm_delta_bpm"].append(
            safe_float(rm.get("delta_bpm", np.nan))
        )
        metrics["rhythm_delta_bpm_folded"].append(
            safe_float(rm.get("delta_bpm_folded", np.nan))
        )
        metrics["rhythm_beat_fmeasure"].append(
            safe_float(rm.get("beat_mir_eval", {}).get("F-measure", np.nan))
        )
        metrics["rhythm_info_gain"].append(
            safe_float(rm.get("beat_mir_eval", {}).get("Information gain", np.nan))
        )

        # ---------------- structure ----------------
        metrics["structure_boundary_f"].append(
            safe_float(s.get("boundary_f_after_shift", s.get("boundary_f", np.nan)))
        )
        metrics["structure_ari"].append(
            safe_float(s.get("ari", np.nan))
        )
        metrics["structure_score"].append(
            safe_float(s.get("structural_form_score", np.nan))
        )

        # ---------------- melody raw ----------------
        m_raw = m.get("raw", {})
        metrics["melody_raw_overall_acc"].append(
            safe_float(m_raw.get("mir_melody", {}).get("overall_accuracy", np.nan))
        )
        metrics["melody_raw_voicing_recall"].append(
            safe_float(m_raw.get("mir_melody", {}).get("voicing_recall", np.nan))
        )
        metrics["melody_raw_pitch_acc"].append(
            safe_float(m_raw.get("mir_melody", {}).get("raw_pitch_accuracy", np.nan))
        )
        metrics["melody_raw_chroma_acc"].append(
            safe_float(m_raw.get("mir_melody", {}).get("raw_chroma_accuracy", np.nan))
        )
        metrics["melody_raw_contour_dtw"].append(
            safe_float(m_raw.get("contour_dtw", {}).get("pitch_dtw_cosine", np.nan))
        )
        metrics["melody_raw_motif_jaccard"].append(
            safe_float(m_raw.get("motif_overlap_n3", {}).get("jaccard", np.nan))
        )
        metrics["melody_raw_motif_recall"].append(
            safe_float(m_raw.get("motif_overlap_n3", {}).get("recall", np.nan))
        )

        # ---------------- melody normalized ----------------
        m_norm = m.get("normalized", {})
        metrics["melody_norm_overall_acc"].append(
            safe_float(m_norm.get("mir_melody", {}).get("overall_accuracy", np.nan))
        )
        metrics["melody_norm_voicing_recall"].append(
            safe_float(m_norm.get("mir_melody", {}).get("voicing_recall", np.nan))
        )
        metrics["melody_norm_pitch_acc"].append(
            safe_float(m_norm.get("mir_melody", {}).get("raw_pitch_accuracy", np.nan))
        )
        metrics["melody_norm_chroma_acc"].append(
            safe_float(m_norm.get("mir_melody", {}).get("raw_chroma_accuracy", np.nan))
        )
        metrics["melody_norm_contour_dtw"].append(
            safe_float(m_norm.get("contour_dtw", {}).get("pitch_dtw_cosine", np.nan))
        )
        metrics["melody_norm_motif_jaccard"].append(
            safe_float(m_norm.get("motif_overlap_n3", {}).get("jaccard", np.nan))
        )
        metrics["melody_norm_motif_recall"].append(
            safe_float(m_norm.get("motif_overlap_n3", {}).get("recall", np.nan))
        )

    return metrics


metrics = extract_metrics(all_results)

# =========================================================
# 4) REPORT
# =========================================================

EXPECTED = {
    "harmony_key_distance": "high / near 1",
    "harmony_chroma_dtw": "drop",
    "harmony_chroma_mean": "drop",
    "harmony_best_shift_chroma_dtw": "stay higher",
    "harmony_best_shift_chroma_mean": "stay higher",
    "harmony_chord_root": "drop",
    "harmony_chord_majmin": "drop",

    "rhythm_delta_bpm": "near 0",
    "rhythm_delta_bpm_folded": "near 0",
    "rhythm_beat_fmeasure": "high",
    "rhythm_info_gain": "high",

    "structure_boundary_f": "high",
    "structure_ari": "high-ish",
    "structure_score": "high-ish",

    "melody_raw_overall_acc": "mixed",
    "melody_raw_voicing_recall": "high",
    "melody_raw_pitch_acc": "drop",
    "melody_raw_chroma_acc": "drop for tritone",
    "melody_raw_contour_dtw": "high",
    "melody_raw_motif_jaccard": "low-mid",
    "melody_raw_motif_recall": "low-mid",

    "melody_norm_overall_acc": "recover",
    "melody_norm_voicing_recall": "high",
    "melody_norm_pitch_acc": "recover",
    "melody_norm_chroma_acc": "recover",
    "melody_norm_contour_dtw": "high",
    "melody_norm_motif_jaccard": "recover some",
    "melody_norm_motif_recall": "recover some",
}

def print_metric_block(title, keys, metrics_dict):
    print(f"\n{title}")
    print("-" * 112)
    print(f"{'Metric':<40} {'Median':>10} {'Std':>10} {'N':>6} {'Expected':>20}")
    print("-" * 112)

    for name in keys:
        med, std, n = summarize(metrics_dict[name])
        exp = EXPECTED.get(name, "")
        if n > 0:
            print(f"{name:<40} {med:>10.4f} {std:>10.4f} {n:>6} {exp:>20}")
        else:
            print(f"{name:<40} {'N/A':>10} {'N/A':>10} {0:>6} {exp:>20}")

print("\n" + "=" * 112)
print(f"PITCH SHIFT (+{PITCH_SHIFT_STEPS} semitones) — {len(all_results)} attempted pairs, {len(metrics['harmony_key_distance'])} valid")
print("=" * 112)

print_metric_block(
    "HARMONY (should change)",
    [
        "harmony_key_distance",
        "harmony_chroma_dtw",
        "harmony_chroma_mean",
        "harmony_best_shift_chroma_dtw",
        "harmony_best_shift_chroma_mean",
        "harmony_chord_root",
        "harmony_chord_majmin",
    ],
    metrics,
)

print_metric_block(
    "RHYTHM (should remain largely stable)",
    [
        "rhythm_delta_bpm",
        "rhythm_delta_bpm_folded",
        "rhythm_beat_fmeasure",
        "rhythm_info_gain",
    ],
    metrics,
)

print_metric_block(
    "STRUCTURE (should remain largely stable)",
    [
        "structure_boundary_f",
        "structure_ari",
        "structure_score",
    ],
    metrics,
)

print_metric_block(
    "MELODY RAW (absolute-pitch-sensitive)",
    [
        "melody_raw_overall_acc",
        "melody_raw_voicing_recall",
        "melody_raw_pitch_acc",
        "melody_raw_chroma_acc",
        "melody_raw_contour_dtw",
        "melody_raw_motif_jaccard",
        "melody_raw_motif_recall",
    ],
    metrics,
)

print_metric_block(
    "MELODY NORMALIZED (transposition-corrected)",
    [
        "melody_norm_overall_acc",
        "melody_norm_voicing_recall",
        "melody_norm_pitch_acc",
        "melody_norm_chroma_acc",
        "melody_norm_contour_dtw",
        "melody_norm_motif_jaccard",
        "melody_norm_motif_recall",
    ],
    metrics,
)

print("\n" + "=" * 112)

valid_examples = [r for r in all_results if "error" not in r]
if valid_examples:
    print(json.dumps(valid_examples[0], indent=2))


Creating pitch-shifted audio: 100%|██████████| 98/98 [02:51<00:00,  1.75s/it]


✓ Created 98 valid pitch-shifted pairs


Evaluating batches:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
# =========================================================
# TIME-STRETCH EDIT + EVAL PIPELINE
# =========================================================
#
# Assumes these are already defined/imported in your Colab:
#   - downloaded  (list of original wav paths)
#   - harmony_score
#   - structural_score_time_stretch
#   - rhythm_score_time_stretch
#   - melody_score_time_stretch_aware
#
# If you have not defined the stretch-aware wrappers yet,
# use the helper definitions included below.
# =========================================================

import os
import json
import numpy as np
import librosa
import soundfile as sf
from tqdm import tqdm

# =========================================================
# HYPERPARAMETERS
# =========================================================

TIME_STRETCH_FACTOR = 1.5               # edited audio becomes 1.5x longer / slower
LIBROSA_STRETCH_RATE = 1.0 / TIME_STRETCH_FACTOR
AUDIO_EDIT_DIR = "audio_edited/time_stretch"
RESULTS_JSON_PATH = "time_stretch_results.json"

EVAL_BATCH_SIZE = 32
SAVE_EVERY_BATCH = True

# =========================================================
# SETUP
# =========================================================

os.makedirs(AUDIO_EDIT_DIR, exist_ok=True)

# =========================================================
# HELPERS
# =========================================================

def safe_float(x, default=np.nan):
    try:
        x = float(x)
        if np.isnan(x) or np.isinf(x):
            return default
        return x
    except Exception:
        return default


def valid_vals(xs):
    out = []
    for x in xs:
        v = safe_float(x)
        if np.isfinite(v):
            out.append(v)
    return out


def summarize(vals):
    vals = valid_vals(vals)
    if not vals:
        return np.nan, np.nan, 0
    return float(np.median(vals)), float(np.std(vals)), len(vals)


def save_json(path, obj):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)


def stretch_to_ref_time(times, stretch_factor):
    arr = np.asarray(times, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return arr
    return arr / float(stretch_factor)


def make_time_stretched_path(src_path: str, out_dir: str = AUDIO_EDIT_DIR) -> str:
    base = os.path.basename(src_path)
    return os.path.join(out_dir, base)

# =========================================================
# OPTIONAL STRETCH-AWARE WRAPPERS
# Use these if you don't already have them defined
# =========================================================

def rhythm_score_time_stretch(audio_ref: str, audio_est: str, stretch_factor: float):
    t0, b0 = tempo_and_beats_librosa(audio_ref)
    t1, b1 = tempo_and_beats_librosa(audio_est)

    b0 = np.asarray(b0, dtype=float)
    b1 = np.asarray(b1, dtype=float)
    b1_norm = stretch_to_ref_time(b1, stretch_factor)

    out = {
        "tempo_ref_bpm": safe_float(t0, 0.0),
        "tempo_est_bpm": safe_float(t1, 0.0),
        "delta_bpm_raw": safe_float(abs(t0 - t1), 0.0),
        "tempo_est_bpm_normalized": safe_float(t1 * stretch_factor, 0.0),
        "delta_bpm_normalized": safe_float(abs(t0 - (t1 * stretch_factor)), 0.0),
    }

    try:
        out["beat_mir_eval_raw"] = beat_fmeasure_mireval(b0, b1)
    except Exception as e:
        out["beat_mir_eval_raw"] = {"error": str(e)}

    try:
        out["beat_mir_eval_normalized"] = beat_fmeasure_mireval(b0, b1_norm)
    except Exception as e:
        out["beat_mir_eval_normalized"] = {"error": str(e)}

    return out


def structural_score_time_stretch(audio_ref: str, audio_est: str, stretch_factor: float):
    out = {}

    try:
        out["raw"] = structural_score(audio_ref, audio_est)
    except Exception as e:
        out["raw"] = {"error": str(e)}

    try:
        rb, rl = librosa_segments(audio_ref)
        eb, el = librosa_segments(audio_est)

        rb = np.asarray(rb, dtype=float)
        eb = np.asarray(eb, dtype=float)
        eb_norm = stretch_to_ref_time(eb, stretch_factor)

        best = best_boundary_shift(rb, eb_norm, window=0.5, search_range=2.0, step=0.02)
        eb_shifted = np.asarray(eb_norm, dtype=float) + best["delta"]
        norm_scores = mir_segment_metrics(rb, eb_shifted, rl, el)

        boundary_f_after_shift = max(0.0, safe_float(best["F"], 0.0))
        ari = safe_float(norm_scores.get("ari", 0.0), 0.0)

        norm_scores.update({
            "best_boundary_shift_sec": safe_float(best["delta"], 0.0),
            "boundary_f_after_shift": boundary_f_after_shift,
            "structural_form_score": safe_float(0.7 * boundary_f_after_shift + 0.3 * ari, 0.0),
        })
        out["normalized"] = norm_scores
    except Exception as e:
        out["normalized"] = {"error": str(e)}

    return out


def melody_score_time_stretch_aware(audio_ref: str, audio_est: str, stretch_factor: float):
    tr, fr, vr = f0_librosa(audio_ref)
    te, fe, ve = f0_librosa(audio_est)

    if len(tr) == 0 or len(te) == 0:
        return {"error": "F0 extraction failed"}

    out = {}

    try:
        out["raw"] = {
            "mir_melody": mir_melody_scores(tr, fr, vr, te, fe, ve),
            "contour_dtw": contour_dtw_distance(tr, fr, te, fe),
            "motif_overlap_n3": motif_ngram_overlap(tr, fr, te, fe, vr, ve, n=3),
        }
    except Exception as e:
        out["raw"] = {"error": str(e)}

    te_norm = stretch_to_ref_time(te, stretch_factor)

    try:
        out["normalized"] = {
            "mir_melody": mir_melody_scores(tr, fr, vr, te_norm, fe, ve),
            "contour_dtw": contour_dtw_distance(tr, fr, te_norm, fe),
            "motif_overlap_n3": motif_ngram_overlap(tr, fr, te_norm, fe, vr, ve, n=3),
        }
    except Exception as e:
        out["normalized"] = {"error": str(e)}

    return out

# =========================================================
# 1) CREATE TIME-STRETCHED COPIES
# =========================================================

time_stretched = []

for src_path in tqdm(downloaded, desc=f"Creating time-stretched audio ({TIME_STRETCH_FACTOR}x longer)"):
    try:
        out_path = make_time_stretched_path(src_path)

        if not os.path.exists(out_path):
            y, sr = librosa.load(src_path, sr=None, mono=True)
            y_stretched = librosa.effects.time_stretch(y, rate=LIBROSA_STRETCH_RATE)
            sf.write(out_path, y_stretched, sr)

        time_stretched.append(out_path)

    except Exception as e:
        print(f"⚠️ Failed time stretch for {src_path}: {e}")
        time_stretched.append(None)

pairs = [(ref, est) for ref, est in zip(downloaded, time_stretched) if est is not None]
print(f"✓ Created {len(pairs)} valid time-stretched pairs")

# =========================================================
# 2) EVALUATE
# =========================================================

def evaluate_pair(ref: str, est: str, stretch_factor: float = TIME_STRETCH_FACTOR) -> dict:
    return {
        "ref": ref,
        "est": est,
        "edit_type": "time_stretch",
        "timeline_stretch_factor": stretch_factor,
        "harmony_tonality": harmony_score(ref, est),
        "rhythm_meter": rhythm_score_time_stretch(ref, est, stretch_factor),
        "structural_form": structural_score_time_stretch(ref, est, stretch_factor),
        "melodic_content": melody_score_time_stretch_aware(ref, est, stretch_factor),
    }


all_results = []

for batch_start in tqdm(range(0, len(pairs), EVAL_BATCH_SIZE), desc="Evaluating batches"):
    batch = pairs[batch_start:batch_start + EVAL_BATCH_SIZE]

    batch_results = []
    for ref, est in batch:
        try:
            batch_results.append(evaluate_pair(ref, est, TIME_STRETCH_FACTOR))
        except Exception as e:
            batch_results.append({
                "ref": ref,
                "est": est,
                "edit_type": "time_stretch",
                "timeline_stretch_factor": TIME_STRETCH_FACTOR,
                "error": str(e),
            })

    all_results.extend(batch_results)

    if SAVE_EVERY_BATCH:
        save_json(RESULTS_JSON_PATH, all_results)

    print(f"✓ batch {batch_start // EVAL_BATCH_SIZE + 1} done ({min(batch_start + EVAL_BATCH_SIZE, len(pairs))}/{len(pairs)})")

save_json(RESULTS_JSON_PATH, all_results)
print(f"✓ Saved raw results to {RESULTS_JSON_PATH}")

# =========================================================
# 3) EXTRACT METRICS
# =========================================================

def extract_metrics(results):
    metrics = {
        # harmony
        "harmony_key_distance": [],
        "harmony_chroma_dtw": [],
        "harmony_chroma_mean": [],
        "harmony_best_shift_chroma_dtw": [],
        "harmony_best_shift_chroma_mean": [],
        "harmony_chord_root": [],
        "harmony_chord_majmin": [],

        # rhythm raw
        "rhythm_delta_bpm_raw": [],
        "rhythm_beat_fmeasure_raw": [],
        "rhythm_info_gain_raw": [],

        # rhythm normalized
        "rhythm_delta_bpm_normalized": [],
        "rhythm_beat_fmeasure_normalized": [],
        "rhythm_info_gain_normalized": [],

        # structure raw
        "structure_boundary_f_raw": [],
        "structure_ari_raw": [],
        "structure_score_raw": [],

        # structure normalized
        "structure_boundary_f_normalized": [],
        "structure_ari_normalized": [],
        "structure_score_normalized": [],

        # melody raw
        "melody_raw_overall_acc": [],
        "melody_raw_voicing_recall": [],
        "melody_raw_pitch_acc": [],
        "melody_raw_chroma_acc": [],
        "melody_raw_contour_dtw": [],
        "melody_raw_motif_jaccard": [],
        "melody_raw_motif_recall": [],

        # melody normalized
        "melody_norm_overall_acc": [],
        "melody_norm_voicing_recall": [],
        "melody_norm_pitch_acc": [],
        "melody_norm_chroma_acc": [],
        "melody_norm_contour_dtw": [],
        "melody_norm_motif_jaccard": [],
        "melody_norm_motif_recall": [],
    }

    for r in results:
        if "error" in r:
            continue

        h = r.get("harmony_tonality", {})
        rm = r.get("rhythm_meter", {})
        s = r.get("structural_form", {})
        m = r.get("melodic_content", {})

        # ---------------- harmony ----------------
        metrics["harmony_key_distance"].append(
            safe_float(h.get("key_relatedness", {}).get("distance_norm_0to1", np.nan))
        )
        metrics["harmony_chroma_dtw"].append(
            safe_float(h.get("chroma_similarity", {}).get("chroma_dtw_cosine", np.nan))
        )
        metrics["harmony_chroma_mean"].append(
            safe_float(h.get("chroma_similarity", {}).get("mean_chroma_cosine", np.nan))
        )
        metrics["harmony_best_shift_chroma_dtw"].append(
            safe_float(h.get("chroma_similarity", {}).get("best_shift_chroma_dtw_cosine", np.nan))
        )
        metrics["harmony_best_shift_chroma_mean"].append(
            safe_float(h.get("chroma_similarity", {}).get("best_shift_mean_chroma_cosine", np.nan))
        )
        metrics["harmony_chord_root"].append(
            safe_float(h.get("chord_similarity", {}).get("mir_root", np.nan))
        )
        metrics["harmony_chord_majmin"].append(
            safe_float(h.get("chord_similarity", {}).get("mir_majmin", np.nan))
        )

        # ---------------- rhythm raw ----------------
        metrics["rhythm_delta_bpm_raw"].append(
            safe_float(rm.get("delta_bpm_raw", np.nan))
        )
        metrics["rhythm_beat_fmeasure_raw"].append(
            safe_float(rm.get("beat_mir_eval_raw", {}).get("F-measure", np.nan))
        )
        metrics["rhythm_info_gain_raw"].append(
            safe_float(rm.get("beat_mir_eval_raw", {}).get("Information gain", np.nan))
        )

        # ---------------- rhythm normalized ----------------
        metrics["rhythm_delta_bpm_normalized"].append(
            safe_float(rm.get("delta_bpm_normalized", np.nan))
        )
        metrics["rhythm_beat_fmeasure_normalized"].append(
            safe_float(rm.get("beat_mir_eval_normalized", {}).get("F-measure", np.nan))
        )
        metrics["rhythm_info_gain_normalized"].append(
            safe_float(rm.get("beat_mir_eval_normalized", {}).get("Information gain", np.nan))
        )

        # ---------------- structure raw ----------------
        s_raw = s.get("raw", {})
        metrics["structure_boundary_f_raw"].append(
            safe_float(s_raw.get("boundary_f_after_shift", s_raw.get("boundary_f", np.nan)))
        )
        metrics["structure_ari_raw"].append(
            safe_float(s_raw.get("ari", np.nan))
        )
        metrics["structure_score_raw"].append(
            safe_float(s_raw.get("structural_form_score", np.nan))
        )

        # ---------------- structure normalized ----------------
        s_norm = s.get("normalized", {})
        metrics["structure_boundary_f_normalized"].append(
            safe_float(s_norm.get("boundary_f_after_shift", s_norm.get("boundary_f", np.nan)))
        )
        metrics["structure_ari_normalized"].append(
            safe_float(s_norm.get("ari", np.nan))
        )
        metrics["structure_score_normalized"].append(
            safe_float(s_norm.get("structural_form_score", np.nan))
        )

        # ---------------- melody raw ----------------
        m_raw = m.get("raw", {})
        metrics["melody_raw_overall_acc"].append(
            safe_float(m_raw.get("mir_melody", {}).get("overall_accuracy", np.nan))
        )
        metrics["melody_raw_voicing_recall"].append(
            safe_float(m_raw.get("mir_melody", {}).get("voicing_recall", np.nan))
        )
        metrics["melody_raw_pitch_acc"].append(
            safe_float(m_raw.get("mir_melody", {}).get("raw_pitch_accuracy", np.nan))
        )
        metrics["melody_raw_chroma_acc"].append(
            safe_float(m_raw.get("mir_melody", {}).get("raw_chroma_accuracy", np.nan))
        )
        metrics["melody_raw_contour_dtw"].append(
            safe_float(m_raw.get("contour_dtw", {}).get("pitch_dtw_cosine", np.nan))
        )
        metrics["melody_raw_motif_jaccard"].append(
            safe_float(m_raw.get("motif_overlap_n3", {}).get("jaccard", np.nan))
        )
        metrics["melody_raw_motif_recall"].append(
            safe_float(m_raw.get("motif_overlap_n3", {}).get("recall", np.nan))
        )

        # ---------------- melody normalized ----------------
        m_norm = m.get("normalized", {})
        metrics["melody_norm_overall_acc"].append(
            safe_float(m_norm.get("mir_melody", {}).get("overall_accuracy", np.nan))
        )
        metrics["melody_norm_voicing_recall"].append(
            safe_float(m_norm.get("mir_melody", {}).get("voicing_recall", np.nan))
        )
        metrics["melody_norm_pitch_acc"].append(
            safe_float(m_norm.get("mir_melody", {}).get("raw_pitch_accuracy", np.nan))
        )
        metrics["melody_norm_chroma_acc"].append(
            safe_float(m_norm.get("mir_melody", {}).get("raw_chroma_accuracy", np.nan))
        )
        metrics["melody_norm_contour_dtw"].append(
            safe_float(m_norm.get("contour_dtw", {}).get("pitch_dtw_cosine", np.nan))
        )
        metrics["melody_norm_motif_jaccard"].append(
            safe_float(m_norm.get("motif_overlap_n3", {}).get("jaccard", np.nan))
        )
        metrics["melody_norm_motif_recall"].append(
            safe_float(m_norm.get("motif_overlap_n3", {}).get("recall", np.nan))
        )

    return metrics


metrics = extract_metrics(all_results)

# =========================================================
# 4) REPORT
# =========================================================

EXPECTED = {
    "harmony_key_distance": "low",
    "harmony_chroma_dtw": "stay high",
    "harmony_chroma_mean": "stay high",
    "harmony_best_shift_chroma_dtw": "stay high",
    "harmony_best_shift_chroma_mean": "stay high",
    "harmony_chord_root": "stay high-ish",
    "harmony_chord_majmin": "stay high-ish",

    "rhythm_delta_bpm_raw": "change",
    "rhythm_beat_fmeasure_raw": "drop",
    "rhythm_info_gain_raw": "drop",

    "rhythm_delta_bpm_normalized": "recover near 0",
    "rhythm_beat_fmeasure_normalized": "recover high",
    "rhythm_info_gain_normalized": "recover high",

    "structure_boundary_f_raw": "mixed / may drop",
    "structure_ari_raw": "mixed",
    "structure_score_raw": "mixed",

    "structure_boundary_f_normalized": "recover high",
    "structure_ari_normalized": "recover",
    "structure_score_normalized": "recover",

    "melody_raw_overall_acc": "drop",
    "melody_raw_voicing_recall": "stay decent",
    "melody_raw_pitch_acc": "drop",
    "melody_raw_chroma_acc": "drop",
    "melody_raw_contour_dtw": "drop some",
    "melody_raw_motif_jaccard": "drop",
    "melody_raw_motif_recall": "drop",

    "melody_norm_overall_acc": "recover",
    "melody_norm_voicing_recall": "stay decent",
    "melody_norm_pitch_acc": "recover",
    "melody_norm_chroma_acc": "recover",
    "melody_norm_contour_dtw": "recover",
    "melody_norm_motif_jaccard": "recover some",
    "melody_norm_motif_recall": "recover some",
}

def print_metric_block(title, keys, metrics_dict):
    print(f"\n{title}")
    print("-" * 112)
    print(f"{'Metric':<40} {'Median':>10} {'Std':>10} {'N':>6} {'Expected':>20}")
    print("-" * 112)

    for name in keys:
        med, std, n = summarize(metrics_dict[name])
        exp = EXPECTED.get(name, "")
        if n > 0:
            print(f"{name:<40} {med:>10.4f} {std:>10.4f} {n:>6} {exp:>20}")
        else:
            print(f"{name:<40} {'N/A':>10} {'N/A':>10} {0:>6} {exp:>20}")

print("\n" + "=" * 112)
print(f"TIME STRETCH ({TIME_STRETCH_FACTOR}x longer) — {len(all_results)} attempted pairs, {len(metrics['harmony_key_distance'])} valid")
print("=" * 112)

print_metric_block(
    "HARMONY (should stay similar)",
    [
        "harmony_key_distance",
        "harmony_chroma_dtw",
        "harmony_chroma_mean",
        "harmony_best_shift_chroma_dtw",
        "harmony_best_shift_chroma_mean",
        "harmony_chord_root",
        "harmony_chord_majmin",
    ],
    metrics,
)

print_metric_block(
    "RHYTHM RAW (absolute-time comparison)",
    [
        "rhythm_delta_bpm_raw",
        "rhythm_beat_fmeasure_raw",
        "rhythm_info_gain_raw",
    ],
    metrics,
)

print_metric_block(
    "RHYTHM NORMALIZED (undo known stretch)",
    [
        "rhythm_delta_bpm_normalized",
        "rhythm_beat_fmeasure_normalized",
        "rhythm_info_gain_normalized",
    ],
    metrics,
)

print_metric_block(
    "STRUCTURE RAW (absolute-time comparison)",
    [
        "structure_boundary_f_raw",
        "structure_ari_raw",
        "structure_score_raw",
    ],
    metrics,
)

print_metric_block(
    "STRUCTURE NORMALIZED (undo known stretch)",
    [
        "structure_boundary_f_normalized",
        "structure_ari_normalized",
        "structure_score_normalized",
    ],
    metrics,
)

print_metric_block(
    "MELODY RAW (absolute-time comparison)",
    [
        "melody_raw_overall_acc",
        "melody_raw_voicing_recall",
        "melody_raw_pitch_acc",
        "melody_raw_chroma_acc",
        "melody_raw_contour_dtw",
        "melody_raw_motif_jaccard",
        "melody_raw_motif_recall",
    ],
    metrics,
)

print_metric_block(
    "MELODY NORMALIZED (undo known stretch)",
    [
        "melody_norm_overall_acc",
        "melody_norm_voicing_recall",
        "melody_norm_pitch_acc",
        "melody_norm_chroma_acc",
        "melody_norm_contour_dtw",
        "melody_norm_motif_jaccard",
        "melody_norm_motif_recall",
    ],
    metrics,
)

print("\n" + "=" * 112)

valid_examples = [r for r in all_results if "error" not in r]
if valid_examples:
    print(json.dumps(valid_examples[0], indent=2))

Creating time-stretched audio (1.5x longer): 100%|██████████| 144/144 [00:16<00:00,  8.96it/s]


✓ Created 144 valid time-stretched pairs


Evaluating batches:   0%|          | 0/5 [00:00<?, ?it/s]/home/eric/MuseCPEval/lib/python3.11/site-packages/numpy/core/_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/eric/MuseCPEval/lib/python3.11/site-packages/numpy/core/_methods.py:198: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/eric/MuseCPEval/lib/python3.11/site-packages/mir_eval/segment.py:120: UserWarning: Estimated intervals are empty.
  warnings.warn("Estimated intervals are empty.")
/home/eric/MuseCPEval/lib/python3.11/site-packages/mir_eval/segment.py:120: UserWarning: Estimated intervals are empty.
  warnings.warn("Estimated intervals are empty.")
/home/eric/MuseCPEval/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/eric/MuseCPEval/lib/python3.11/site-packages/numpy/core/_metho

✓ batch 1 done (32/144)


/home/eric/MuseCPEval/lib/python3.11/site-packages/mir_eval/segment.py:120: UserWarning: Estimated intervals are empty.
  warnings.warn("Estimated intervals are empty.")
/home/eric/MuseCPEval/lib/python3.11/site-packages/mir_eval/segment.py:117: UserWarning: Reference intervals are empty.
  warnings.warn("Reference intervals are empty.")
/home/eric/MuseCPEval/lib/python3.11/site-packages/mir_eval/segment.py:117: UserWarning: Reference intervals are empty.
  warnings.warn("Reference intervals are empty.")
/home/eric/MuseCPEval/lib/python3.11/site-packages/mir_eval/segment.py:117: UserWarning: Reference intervals are empty.
  warnings.warn("Reference intervals are empty.")
/home/eric/MuseCPEval/lib/python3.11/site-packages/mir_eval/segment.py:120: UserWarning: Estimated intervals are empty.
  warnings.warn("Estimated intervals are empty.")
/home/eric/MuseCPEval/lib/python3.11/site-packages/numpy/core/_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=

...

✓ batch 3 done (96/144)


/home/eric/MuseCPEval/lib/python3.11/site-packages/mir_eval/beat.py:91: UserWarning: Reference beats are empty.
  warnings.warn("Reference beats are empty.")
/home/eric/MuseCPEval/lib/python3.11/site-packages/mir_eval/segment.py:120: UserWarning: Estimated intervals are empty.
  warnings.warn("Estimated intervals are empty.")
/home/eric/MuseCPEval/lib/python3.11/site-packages/mir_eval/segment.py:117: UserWarning: Reference intervals are empty.
  warnings.warn("Reference intervals are empty.")
/home/eric/MuseCPEval/lib/python3.11/site-packages/mir_eval/segment.py:120: UserWarning: Estimated intervals are empty.
  warnings.warn("Estimated intervals are empty.")
/home/eric/MuseCPEval/lib/python3.11/site-packages/numpy/core/_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/eric/MuseCPEval/lib/python3.11/site-packages/numpy/core/_methods.py:198: RuntimeWarning: invalid value encountered in scalar divide
  ret

✓ batch 4 done (128/144)


/home/eric/MuseCPEval/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/eric/MuseCPEval/lib/python3.11/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/eric/MuseCPEval/lib/python3.11/site-packages/mir_eval/segment.py:117: UserWarning: Reference intervals are empty.
  warnings.warn("Reference intervals are empty.")
/home/eric/MuseCPEval/lib/python3.11/site-packages/mir_eval/segment.py:117: UserWarning: Reference intervals are empty.
  warnings.warn("Reference intervals are empty.")
/home/eric/MuseCPEval/lib/python3.11/site-packages/mir_eval/segment.py:120: UserWarning: Estimated intervals are empty.
  warnings.warn("Estimated intervals are empty.")
/home/eric/MuseCPEval/lib/python3.11/site-packages/mir_eval/segment.py:117: UserWarning: Reference intervals are empty.
  warnings.warn("Referen

✓ batch 5 done (144/144)
✓ Saved raw results to time_stretch_results.json

TIME STRETCH (1.5x longer) — 144 attempted pairs, 144 valid

HARMONY (should stay similar)
----------------------------------------------------------------------------------------------------------------
Metric                                       Median        Std      N             Expected
----------------------------------------------------------------------------------------------------------------
harmony_key_distance                         0.0000     0.1624    144                  low
harmony_chroma_dtw                           0.9852     0.0048    144            stay high
harmony_chroma_mean                          0.9978     0.0028    144            stay high
harmony_best_shift_chroma_dtw                0.9852     0.0048    144            stay high
harmony_best_shift_chroma_mean               0.9978     0.0027    144            stay high
harmony_chord_root                           0.3350     0.1643

In [ ]:
# =========================================================
# SEGMENT-SHUFFLE EDIT + EVAL PIPELINE (DETERMINISTIC, NO SKIPS)
# =========================================================

import os
import json
import numpy as np
import librosa
import soundfile as sf
from tqdm import tqdm

# =========================================================
# HYPERPARAMETERS
# =========================================================

AUDIO_EDIT_DIR = "audio_edited/segment_shuffle"
RESULTS_JSON_PATH = "segment_shuffle_results.json"

EVAL_BATCH_SIZE = 32
SAVE_EVERY_BATCH = True

NUM_SEGMENTS = 4
ROTATE_BY = 1   # guaranteed non-identity if NUM_SEGMENTS >= 2

# =========================================================
# SETUP
# =========================================================

os.makedirs(AUDIO_EDIT_DIR, exist_ok=True)

# =========================================================
# HELPERS
# =========================================================

def safe_float(x, default=np.nan):
    try:
        x = float(x)
        if np.isnan(x) or np.isinf(x):
            return default
        return x
    except Exception:
        return default


def valid_vals(xs):
    out = []
    for x in xs:
        v = safe_float(x)
        if np.isfinite(v):
            out.append(v)
    return out


def summarize(vals):
    vals = valid_vals(vals)
    if not vals:
        return np.nan, np.nan, 0
    return float(np.median(vals)), float(np.std(vals)), len(vals)


def save_json(path, obj):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)


def make_segment_shuffle_path(src_path: str, out_dir: str = AUDIO_EDIT_DIR) -> str:
    return os.path.join(out_dir, os.path.basename(src_path))


def split_audio_into_equal_segments(y: np.ndarray, num_segments: int):
    y = np.asarray(y)
    total_len = len(y)

    if total_len == 0:
        return []

    # robust split: np.array_split never drops samples
    segments = np.array_split(y, num_segments)

    # remove truly empty segments only
    segments = [seg for seg in segments if len(seg) > 0]
    return segments


def deterministic_shuffle_segments(y: np.ndarray, num_segments: int, rotate_by: int = ROTATE_BY):
    segments = split_audio_into_equal_segments(y, num_segments)

    if len(segments) < 2:
        return None, None

    n = len(segments)
    shift = rotate_by % n
    if shift == 0:
        shift = 1

    order = list(range(n))
    shuffled_order = order[shift:] + order[:shift]

    y_shuffled = np.concatenate([segments[i] for i in shuffled_order], axis=0)
    return y_shuffled, shuffled_order

# =========================================================
# 1) CREATE SEGMENT-SHUFFLED COPIES
# =========================================================

segment_shuffled = []
shuffle_orders = []

for src_path in tqdm(downloaded, desc=f"Creating segment-shuffled audio ({NUM_SEGMENTS} equal chunks)"):
    try:
        out_path = make_segment_shuffle_path(src_path)

        y, sr = librosa.load(src_path, sr=None, mono=True)

        y_shuffled, shuffled_order = deterministic_shuffle_segments(
            y,
            num_segments=NUM_SEGMENTS,
            rotate_by=ROTATE_BY,
        )

        if y_shuffled is None:
            print(f"⚠️ Skipping {src_path}: audio too short or empty")
            segment_shuffled.append(None)
            shuffle_orders.append(None)
            continue

        sf.write(out_path, y_shuffled, sr)
        segment_shuffled.append(out_path)
        shuffle_orders.append(shuffled_order)

    except Exception as e:
        print(f"⚠️ Failed segment shuffle for {src_path}: {e}")
        segment_shuffled.append(None)
        shuffle_orders.append(None)

pairs = [
    (ref, est, order)
    for ref, est, order in zip(downloaded, segment_shuffled, shuffle_orders)
    if est is not None and order is not None
]

print(f"✓ Created {len(pairs)} valid segment-shuffled pairs")

# =========================================================
# 2) EVALUATE
# =========================================================

def evaluate_pair(ref: str, est: str, order) -> dict:
    return {
        "ref": ref,
        "est": est,
        "edit_type": "segment_shuffle",
        "num_segments": NUM_SEGMENTS,
        "shuffle_order": order,
        "harmony_tonality": harmony_score(ref, est),
        "rhythm_meter": rhythm_score(ref, est),
        "structural_form": structural_score(ref, est),
        "melodic_content": melody_score(ref, est),
    }


all_results = []

for batch_start in tqdm(range(0, len(pairs), EVAL_BATCH_SIZE), desc="Evaluating batches"):
    batch = pairs[batch_start:batch_start + EVAL_BATCH_SIZE]

    batch_results = []
    for ref, est, order in batch:
        try:
            batch_results.append(evaluate_pair(ref, est, order))
        except Exception as e:
            batch_results.append({
                "ref": ref,
                "est": est,
                "edit_type": "segment_shuffle",
                "num_segments": NUM_SEGMENTS,
                "shuffle_order": order,
                "error": str(e),
            })

    all_results.extend(batch_results)

    if SAVE_EVERY_BATCH:
        save_json(RESULTS_JSON_PATH, all_results)

    print(f"✓ batch {batch_start // EVAL_BATCH_SIZE + 1} done ({min(batch_start + EVAL_BATCH_SIZE, len(pairs))}/{len(pairs)})")

save_json(RESULTS_JSON_PATH, all_results)
print(f"✓ Saved raw results to {RESULTS_JSON_PATH}")

# =========================================================
# 3) EXTRACT METRICS
# =========================================================

def extract_metrics(results):
    metrics = {
        "harmony_key_distance": [],
        "harmony_chroma_dtw": [],
        "harmony_chroma_mean": [],
        "harmony_best_shift_chroma_dtw": [],
        "harmony_best_shift_chroma_mean": [],
        "harmony_chord_root": [],
        "harmony_chord_majmin": [],

        "rhythm_delta_bpm": [],
        "rhythm_delta_bpm_folded": [],
        "rhythm_beat_fmeasure": [],
        "rhythm_info_gain": [],

        "structure_boundary_f": [],
        "structure_ari": [],
        "structure_score": [],

        "melody_overall_acc": [],
        "melody_voicing_recall": [],
        "melody_pitch_acc": [],
        "melody_chroma_acc": [],
        "melody_contour_dtw": [],
        "melody_motif_jaccard": [],
        "melody_motif_recall": [],
    }

    for r in results:
        if "error" in r:
            continue

        h = r.get("harmony_tonality", {})
        rm = r.get("rhythm_meter", {})
        s = r.get("structural_form", {})
        m = r.get("melodic_content", {})

        metrics["harmony_key_distance"].append(safe_float(h.get("key_relatedness", {}).get("distance_norm_0to1", np.nan)))
        metrics["harmony_chroma_dtw"].append(safe_float(h.get("chroma_similarity", {}).get("chroma_dtw_cosine", np.nan)))
        metrics["harmony_chroma_mean"].append(safe_float(h.get("chroma_similarity", {}).get("mean_chroma_cosine", np.nan)))
        metrics["harmony_best_shift_chroma_dtw"].append(safe_float(h.get("chroma_similarity", {}).get("best_shift_chroma_dtw_cosine", np.nan)))
        metrics["harmony_best_shift_chroma_mean"].append(safe_float(h.get("chroma_similarity", {}).get("best_shift_mean_chroma_cosine", np.nan)))
        metrics["harmony_chord_root"].append(safe_float(h.get("chord_similarity", {}).get("mir_root", np.nan)))
        metrics["harmony_chord_majmin"].append(safe_float(h.get("chord_similarity", {}).get("mir_majmin", np.nan)))

        metrics["rhythm_delta_bpm"].append(safe_float(rm.get("delta_bpm", np.nan)))
        metrics["rhythm_delta_bpm_folded"].append(safe_float(rm.get("delta_bpm_folded", np.nan)))
        metrics["rhythm_beat_fmeasure"].append(safe_float(rm.get("beat_mir_eval", {}).get("F-measure", np.nan)))
        metrics["rhythm_info_gain"].append(safe_float(rm.get("beat_mir_eval", {}).get("Information gain", np.nan)))

        metrics["structure_boundary_f"].append(safe_float(s.get("boundary_f_after_shift", s.get("boundary_f", np.nan))))
        metrics["structure_ari"].append(safe_float(s.get("ari", np.nan)))
        metrics["structure_score"].append(safe_float(s.get("structural_form_score", np.nan)))

        metrics["melody_overall_acc"].append(safe_float(m.get("mir_melody", {}).get("overall_accuracy", np.nan)))
        metrics["melody_voicing_recall"].append(safe_float(m.get("mir_melody", {}).get("voicing_recall", np.nan)))
        metrics["melody_pitch_acc"].append(safe_float(m.get("mir_melody", {}).get("raw_pitch_accuracy", np.nan)))
        metrics["melody_chroma_acc"].append(safe_float(m.get("mir_melody", {}).get("raw_chroma_accuracy", np.nan)))
        metrics["melody_contour_dtw"].append(safe_float(m.get("contour_dtw", {}).get("pitch_dtw_cosine", np.nan)))
        metrics["melody_motif_jaccard"].append(safe_float(m.get("motif_overlap_n3", {}).get("jaccard", np.nan)))
        metrics["melody_motif_recall"].append(safe_float(m.get("motif_overlap_n3", {}).get("recall", np.nan)))

    return metrics


metrics = extract_metrics(all_results)

# =========================================================
# 4) REPORT
# =========================================================

EXPECTED = {
    "harmony_key_distance": "stay low-ish",
    "harmony_chroma_dtw": "stay moderate/high",
    "harmony_chroma_mean": "stay moderate/high",
    "harmony_best_shift_chroma_dtw": "stay moderate/high",
    "harmony_best_shift_chroma_mean": "stay moderate/high",
    "harmony_chord_root": "stay moderate/high",
    "harmony_chord_majmin": "stay moderate/high",

    "rhythm_delta_bpm": "near 0-ish",
    "rhythm_delta_bpm_folded": "near 0-ish",
    "rhythm_beat_fmeasure": "may drop some",
    "rhythm_info_gain": "may drop some",

    "structure_boundary_f": "drop / mixed",
    "structure_ari": "drop strongly",
    "structure_score": "drop strongly",

    "melody_overall_acc": "mixed",
    "melody_voicing_recall": "stay decent",
    "melody_pitch_acc": "mixed",
    "melody_chroma_acc": "mixed",
    "melody_contour_dtw": "mixed",
    "melody_motif_jaccard": "mixed",
    "melody_motif_recall": "mixed",
}

def print_metric_block(title, keys, metrics_dict):
    print(f"\n{title}")
    print("-" * 112)
    print(f"{'Metric':<40} {'Median':>10} {'Std':>10} {'N':>6} {'Expected':>20}")
    print("-" * 112)

    for name in keys:
        med, std, n = summarize(metrics_dict[name])
        exp = EXPECTED.get(name, "")
        if n > 0:
            print(f"{name:<40} {med:>10.4f} {std:>10.4f} {n:>6} {exp:>20}")
        else:
            print(f"{name:<40} {'N/A':>10} {'N/A':>10} {0:>6} {exp:>20}")

print("\n" + "=" * 112)
print(f"SEGMENT SHUFFLE ({NUM_SEGMENTS} equal chunks) — {len(all_results)} attempted pairs, {len(metrics['structure_boundary_f'])} valid")
print("=" * 112)

print_metric_block(
    "HARMONY (should stay comparatively more stable)",
    [
        "harmony_key_distance",
        "harmony_chroma_dtw",
        "harmony_chroma_mean",
        "harmony_best_shift_chroma_dtw",
        "harmony_best_shift_chroma_mean",
        "harmony_chord_root",
        "harmony_chord_majmin",
    ],
    metrics,
)

print_metric_block(
    "RHYTHM (may drift due to hard cuts)",
    [
        "rhythm_delta_bpm",
        "rhythm_delta_bpm_folded",
        "rhythm_beat_fmeasure",
        "rhythm_info_gain",
    ],
    metrics,
)

print_metric_block(
    "STRUCTURE (target facet: should change most)",
    [
        "structure_boundary_f",
        "structure_ari",
        "structure_score",
    ],
    metrics,
)

print_metric_block(
    "MELODY (secondary effects possible)",
    [
        "melody_overall_acc",
        "melody_voicing_recall",
        "melody_pitch_acc",
        "melody_chroma_acc",
        "melody_contour_dtw",
        "melody_motif_jaccard",
        "melody_motif_recall",
    ],
    metrics,
)

print("\n" + "=" * 112)

valid_examples = [r for r in all_results if "error" not in r]
if valid_examples:
    print(json.dumps(valid_examples[0], indent=2))

Creating segment-shuffled audio (4 equal chunks): 100%|██████████| 144/144 [00:01<00:00, 91.83it/s]


✓ Created 144 valid segment-shuffled pairs


Evaluating batches:   0%|          | 0/5 [00:00<?, ?it/s]/home/eric/MuseCPEval/lib/python3.11/site-packages/numpy/core/_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/eric/MuseCPEval/lib/python3.11/site-packages/numpy/core/_methods.py:198: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/eric/MuseCPEval/lib/python3.11/site-packages/mir_eval/segment.py:120: UserWarning: Estimated intervals are empty.
  warnings.warn("Estimated intervals are empty.")
/home/eric/MuseCPEval/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/eric/MuseCPEval/lib/python3.11/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/eric/MuseCPEval/lib/python3.11/site-packages/mir_eval/seg

✓ batch 1 done (32/144)


/home/eric/MuseCPEval/lib/python3.11/site-packages/numpy/core/_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/eric/MuseCPEval/lib/python3.11/site-packages/numpy/core/_methods.py:198: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/eric/MuseCPEval/lib/python3.11/site-packages/mir_eval/segment.py:117: UserWarning: Reference intervals are empty.
  warnings.warn("Reference intervals are empty.")
/home/eric/MuseCPEval/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/eric/MuseCPEval/lib/python3.11/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/eric/MuseCPEval/lib/python3.11/site-packages/mir_eval/segment.py:117: UserWarning: Reference intervals are empty.


...

✓ batch 3 done (96/144)


/home/eric/MuseCPEval/lib/python3.11/site-packages/mir_eval/beat.py:91: UserWarning: Reference beats are empty.
  warnings.warn("Reference beats are empty.")
/home/eric/MuseCPEval/lib/python3.11/site-packages/mir_eval/segment.py:120: UserWarning: Estimated intervals are empty.
  warnings.warn("Estimated intervals are empty.")
/home/eric/MuseCPEval/lib/python3.11/site-packages/mir_eval/segment.py:117: UserWarning: Reference intervals are empty.
  warnings.warn("Reference intervals are empty.")
/home/eric/MuseCPEval/lib/python3.11/site-packages/mir_eval/segment.py:120: UserWarning: Estimated intervals are empty.
  warnings.warn("Estimated intervals are empty.")
/home/eric/MuseCPEval/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/eric/MuseCPEval/lib/python3.11/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(

✓ batch 4 done (128/144)


/home/eric/MuseCPEval/lib/python3.11/site-packages/mir_eval/segment.py:117: UserWarning: Reference intervals are empty.
  warnings.warn("Reference intervals are empty.")
/home/eric/MuseCPEval/lib/python3.11/site-packages/mir_eval/segment.py:120: UserWarning: Estimated intervals are empty.
  warnings.warn("Estimated intervals are empty.")
/home/eric/MuseCPEval/lib/python3.11/site-packages/mir_eval/segment.py:117: UserWarning: Reference intervals are empty.
  warnings.warn("Reference intervals are empty.")
/home/eric/MuseCPEval/lib/python3.11/site-packages/mir_eval/segment.py:120: UserWarning: Estimated intervals are empty.
  warnings.warn("Estimated intervals are empty.")
/home/eric/MuseCPEval/lib/python3.11/site-packages/mir_eval/beat.py:376: UserWarning: Only one estimated beat was provided, so beat intervals cannot be computed.
  warnings.warn("Only one estimated beat was provided, so beat intervals"
/home/eric/MuseCPEval/lib/python3.11/site-packages/mir_eval/beat.py:464: UserWarning

✓ batch 5 done (144/144)
✓ Saved raw results to segment_shuffle_results.json

SEGMENT SHUFFLE (4 equal chunks) — 144 attempted pairs, 144 valid

HARMONY (should stay comparatively more stable)
----------------------------------------------------------------------------------------------------------------
Metric                                       Median        Std      N             Expected
----------------------------------------------------------------------------------------------------------------
harmony_key_distance                         0.0000     0.1011    144         stay low-ish
harmony_chroma_dtw                           0.9633     0.0285    144   stay moderate/high
harmony_chroma_mean                          0.9999     0.0007    144   stay moderate/high
harmony_best_shift_chroma_dtw                0.9633     0.0284    144   stay moderate/high
harmony_best_shift_chroma_mean               0.9999     0.0007    144   stay moderate/high
harmony_chord_root                 

In [ ]:
# =========================================================
# VOCAL-ONLY PITCH-SHIFT EDIT + EVAL PIPELINE
# =========================================================
#
# Assumes these are already defined/imported in your Colab:
#   - downloaded   (list of mixture wav paths, e.g. musdb_wav/train_*.wav or music/lmd_100_samples_wav/sample_*.wav)
#   - harmony_score
#   - rhythm_score
#   - structural_score
#   - melody_score
#
# Example imports:
# from harmony_tonality import harmony_score
# from rhythm_meter import rhythm_score
# from structural_form import structural_score
# from melody_motifs import melody_score
#
# This creates a new mixture:
#   shifted_vocals + untouched_accompaniment
# then evaluates it against the original mixture.
# =========================================================

import os
import json
import numpy as np
import musdb
import librosa
import soundfile as sf
from tqdm import tqdm

# =========================================================
# HYPERPARAMETERS
# =========================================================

ROOT_DIR = "musdb"
AUDIO_EDIT_DIR = "audio_edited/vocal_only_pitch_shift"
RESULTS_JSON_PATH = "vocal_only_pitch_shift_results.json"

VOCAL_PITCH_SHIFT_STEPS = 5
EVAL_BATCH_SIZE = 32
SAVE_EVERY_BATCH = True
PRINT_ONE_EXAMPLE = True

# =========================================================
# SETUP
# =========================================================

os.makedirs(AUDIO_EDIT_DIR, exist_ok=True)

# =========================================================
# HELPERS
# =========================================================

def safe_float(x, default=np.nan):
    try:
        x = float(x)
        if np.isnan(x) or np.isinf(x):
            return default
        return x
    except Exception:
        return default


def valid_vals(xs):
    out = []
    for x in xs:
        v = safe_float(x)
        if np.isfinite(v):
            out.append(v)
    return out


def summarize(vals):
    vals = valid_vals(vals)
    if not vals:
        return np.nan, np.nan, 0
    return float(np.median(vals)), float(np.std(vals)), len(vals)


def save_json(path, obj):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)


def get_subset_and_track_name(wav_path: str):
    """
    Expects basename like:
      train_A Classic Education - NightOwl.wav
      test_AM Contra - Heart Peripheral.wav
    """
    base = os.path.basename(wav_path)
    stem = os.path.splitext(base)[0]

    if stem.startswith("train_"):
        return "train", stem[len("train_"):]
    if stem.startswith("test_"):
        return "test", stem[len("test_"):]

    return None, stem


def make_edit_path(src_path: str, out_dir: str = AUDIO_EDIT_DIR) -> str:
    return os.path.join(out_dir, os.path.basename(src_path))


def using_lmd_sample_wavs(paths) -> bool:
    """True when `downloaded` comes from music/lmd_100_samples_wav (sample_*.wav)."""
    if not paths:
        return False
    for p in paths:
        base = os.path.basename(p)
        if not (base.startswith("sample_") and base.lower().endswith(".wav")):
            return False
    return True


def build_track_lookup(root_dir: str):
    db_train = musdb.DB(root=root_dir, subsets="train", download=True)
    db_test = musdb.DB(root=root_dir, subsets="test", download=True)

    lookup = {}
    for track in db_train.tracks:
        lookup[("train", track.name)] = track
    for track in db_test.tracks:
        lookup[("test", track.name)] = track

    return lookup


def to_mono_1d(audio: np.ndarray) -> np.ndarray:
    audio = np.asarray(audio, dtype=np.float32)

    if audio.ndim == 1:
        return audio

    # MUSDB audio is typically [samples, channels]
    if audio.ndim == 2:
        if audio.shape[1] == 1:
            return audio[:, 0]
        return np.mean(audio, axis=1)

    raise ValueError(f"Unsupported audio shape: {audio.shape}")


def match_length(x: np.ndarray, target_len: int) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32).reshape(-1)

    if len(x) == target_len:
        return x
    if len(x) > target_len:
        return x[:target_len]

    out = np.zeros((target_len,), dtype=np.float32)
    out[:len(x)] = x
    return out


def get_target_audio(track, key: str):
    if not hasattr(track, "targets") or key not in track.targets:
        return None
    return np.asarray(track.targets[key].audio, dtype=np.float32)


def get_vocals_and_accompaniment(track):
    vocals = get_target_audio(track, "vocals")
    accompaniment = None

    if hasattr(track, "targets") and "accompaniment" in track.targets:
        accompaniment = np.asarray(track.targets["accompaniment"].audio, dtype=np.float32)
    else:
        parts = []
        for key in ["drums", "bass", "other"]:
            part = get_target_audio(track, key)
            if part is not None:
                parts.append(part)
        if len(parts) > 0:
            accompaniment = parts[0].copy()
            for p in parts[1:]:
                accompaniment = accompaniment + p

    return vocals, accompaniment


def remix_vocal_pitch_shift(track, n_steps: int):
    vocals_stereo, accompaniment_stereo = get_vocals_and_accompaniment(track)
    if vocals_stereo is None or accompaniment_stereo is None:
        return None

    sr = int(track.rate)

    vocals_mono = to_mono_1d(vocals_stereo)
    accompaniment_mono = to_mono_1d(accompaniment_stereo)

    vocals_shifted = librosa.effects.pitch_shift(vocals_mono, sr=sr, n_steps=n_steps)

    target_len = max(len(vocals_shifted), len(accompaniment_mono))
    vocals_shifted = match_length(vocals_shifted, target_len)
    accompaniment_mono = match_length(accompaniment_mono, target_len)

    remix = vocals_shifted + accompaniment_mono
    remix = remix.astype(np.float32)

    # light safety normalization
    peak = np.max(np.abs(remix)) if len(remix) else 0.0
    if peak > 1.0:
        remix = remix / peak

    return remix, sr


# =========================================================
# 1) BUILD MUSDB LOOKUP (skipped for LMD sample WAVs)
# =========================================================

if using_lmd_sample_wavs(downloaded):
    TRACK_LOOKUP = {}
    print(
        "✓ LMD sample WAVs: using full-mix pitch shift here (MUSDB stems / vocal-only split unavailable)."
    )
else:
    TRACK_LOOKUP = build_track_lookup(ROOT_DIR)
    print(f"✓ Built MUSDB lookup with {len(TRACK_LOOKUP)} tracks")

# =========================================================
# 2) CREATE VOCAL-ONLY PITCH-SHIFTED MIXTURES
# =========================================================

edited_paths = []

for src_path in tqdm(downloaded, desc=f"Creating vocal-only pitch-shift audio (+{VOCAL_PITCH_SHIFT_STEPS} st)"):
    try:
        subset, track_name = get_subset_and_track_name(src_path)
        out_path = make_edit_path(src_path)

        if os.path.exists(out_path):
            edited_paths.append(out_path)
            continue

        if using_lmd_sample_wavs(downloaded):
            y, sr = librosa.load(src_path, sr=None, mono=True)
            y_edit = librosa.effects.pitch_shift(y, sr=sr, n_steps=VOCAL_PITCH_SHIFT_STEPS)
            sf.write(out_path, np.asarray(y_edit, dtype=np.float32), sr)
            edited_paths.append(out_path)
            continue

        track = TRACK_LOOKUP.get((subset, track_name))
        if track is None:
            print(f"⚠️ Could not match MUSDB track for {src_path}")
            edited_paths.append(None)
            continue

        remix = remix_vocal_pitch_shift(track, VOCAL_PITCH_SHIFT_STEPS)
        if remix is None:
            print(f"⚠️ Could not build remix for {src_path}")
            edited_paths.append(None)
            continue

        y_edit, sr = remix
        sf.write(out_path, y_edit, sr)
        edited_paths.append(out_path)

    except Exception as e:
        print(f"⚠️ Failed vocal-only pitch shift for {src_path}: {e}")
        edited_paths.append(None)

pairs = [(ref, est) for ref, est in zip(downloaded, edited_paths) if est is not None]
print(f"✓ Created {len(pairs)} valid vocal-only pitch-shift pairs")

# =========================================================
# 3) EVALUATE
# =========================================================

def evaluate_pair(ref: str, est: str) -> dict:
    return {
        "ref": ref,
        "est": est,
        "edit_type": "vocal_only_pitch_shift",
        "vocal_pitch_shift_steps": VOCAL_PITCH_SHIFT_STEPS,
        "harmony_tonality": harmony_score(ref, est),
        "rhythm_meter": rhythm_score(ref, est),
        "structural_form": structural_score(ref, est),
        "melodic_content": melody_score(ref, est),
    }


all_results = []

for batch_start in tqdm(range(0, len(pairs), EVAL_BATCH_SIZE), desc="Evaluating batches"):
    batch = pairs[batch_start:batch_start + EVAL_BATCH_SIZE]

    batch_results = []
    for ref, est in batch:
        try:
            batch_results.append(evaluate_pair(ref, est))
        except Exception as e:
            batch_results.append({
                "ref": ref,
                "est": est,
                "edit_type": "vocal_only_pitch_shift",
                "vocal_pitch_shift_steps": VOCAL_PITCH_SHIFT_STEPS,
                "error": str(e),
            })

    all_results.extend(batch_results)

    if SAVE_EVERY_BATCH:
        save_json(RESULTS_JSON_PATH, all_results)

    print(f"✓ batch {batch_start // EVAL_BATCH_SIZE + 1} done ({min(batch_start + EVAL_BATCH_SIZE, len(pairs))}/{len(pairs)})")

save_json(RESULTS_JSON_PATH, all_results)
print(f"✓ Saved raw results to {RESULTS_JSON_PATH}")

# =========================================================
# 4) EXTRACT METRICS
# =========================================================

def extract_metrics(results):
    metrics = {
        # harmony
        "harmony_key_distance": [],
        "harmony_chroma_dtw": [],
        "harmony_chroma_mean": [],
        "harmony_best_shift_chroma_dtw": [],
        "harmony_best_shift_chroma_mean": [],
        "harmony_chord_root": [],
        "harmony_chord_majmin": [],

        # rhythm
        "rhythm_delta_bpm": [],
        "rhythm_delta_bpm_folded": [],
        "rhythm_beat_fmeasure": [],
        "rhythm_info_gain": [],

        # structure
        "structure_boundary_f": [],
        "structure_ari": [],
        "structure_score": [],

        # melody
        "melody_overall_acc": [],
        "melody_voicing_recall": [],
        "melody_voicing_false_alarm": [],
        "melody_pitch_acc": [],
        "melody_chroma_acc": [],
        "melody_contour_dtw": [],
        "melody_motif_jaccard": [],
        "melody_motif_recall": [],
    }

    for r in results:
        if "error" in r:
            continue

        h = r.get("harmony_tonality", {})
        rm = r.get("rhythm_meter", {})
        s = r.get("structural_form", {})
        m = r.get("melodic_content", {})

        metrics["harmony_key_distance"].append(
            safe_float(h.get("key_relatedness", {}).get("distance_norm_0to1", np.nan))
        )
        metrics["harmony_chroma_dtw"].append(
            safe_float(h.get("chroma_similarity", {}).get("chroma_dtw_cosine", np.nan))
        )
        metrics["harmony_chroma_mean"].append(
            safe_float(h.get("chroma_similarity", {}).get("mean_chroma_cosine", np.nan))
        )
        metrics["harmony_best_shift_chroma_dtw"].append(
            safe_float(h.get("chroma_similarity", {}).get("best_shift_chroma_dtw_cosine", np.nan))
        )
        metrics["harmony_best_shift_chroma_mean"].append(
            safe_float(h.get("chroma_similarity", {}).get("best_shift_mean_chroma_cosine", np.nan))
        )
        metrics["harmony_chord_root"].append(
            safe_float(h.get("chord_similarity", {}).get("mir_root", np.nan))
        )
        metrics["harmony_chord_majmin"].append(
            safe_float(h.get("chord_similarity", {}).get("mir_majmin", np.nan))
        )

        metrics["rhythm_delta_bpm"].append(
            safe_float(rm.get("delta_bpm", np.nan))
        )
        metrics["rhythm_delta_bpm_folded"].append(
            safe_float(rm.get("delta_bpm_folded", np.nan))
        )
        metrics["rhythm_beat_fmeasure"].append(
            safe_float(rm.get("beat_mir_eval", {}).get("F-measure", np.nan))
        )
        metrics["rhythm_info_gain"].append(
            safe_float(rm.get("beat_mir_eval", {}).get("Information gain", np.nan))
        )

        metrics["structure_boundary_f"].append(
            safe_float(s.get("boundary_f_after_shift", s.get("boundary_f", np.nan)))
        )
        metrics["structure_ari"].append(
            safe_float(s.get("ari", np.nan))
        )
        metrics["structure_score"].append(
            safe_float(s.get("structural_form_score", np.nan))
        )

        metrics["melody_overall_acc"].append(
            safe_float(m.get("mir_melody", {}).get("overall_accuracy", np.nan))
        )
        metrics["melody_voicing_recall"].append(
            safe_float(m.get("mir_melody", {}).get("voicing_recall", np.nan))
        )
        metrics["melody_voicing_false_alarm"].append(
            safe_float(m.get("mir_melody", {}).get("voicing_false_alarm", np.nan))
        )
        metrics["melody_pitch_acc"].append(
            safe_float(m.get("mir_melody", {}).get("raw_pitch_accuracy", np.nan))
        )
        metrics["melody_chroma_acc"].append(
            safe_float(m.get("mir_melody", {}).get("raw_chroma_accuracy", np.nan))
        )
        metrics["melody_contour_dtw"].append(
            safe_float(m.get("contour_dtw", {}).get("pitch_dtw_cosine", np.nan))
        )
        metrics["melody_motif_jaccard"].append(
            safe_float(m.get("motif_overlap_n3", {}).get("jaccard", np.nan))
        )
        metrics["melody_motif_recall"].append(
            safe_float(m.get("motif_overlap_n3", {}).get("recall", np.nan))
        )

    return metrics


metrics = extract_metrics(all_results)

# =========================================================
# 5) REPORT
# =========================================================

EXPECTED = {
    "harmony_key_distance": "small/moderate",
    "harmony_chroma_dtw": "stay fairly high",
    "harmony_chroma_mean": "stay fairly high",
    "harmony_best_shift_chroma_dtw": "stay high",
    "harmony_best_shift_chroma_mean": "stay high",
    "harmony_chord_root": "stay moderate/high",
    "harmony_chord_majmin": "stay moderate/high",

    "rhythm_delta_bpm": "near 0-ish",
    "rhythm_delta_bpm_folded": "near 0-ish",
    "rhythm_beat_fmeasure": "stay fairly high",
    "rhythm_info_gain": "stay fairly high",

    "structure_boundary_f": "stay fairly high",
    "structure_ari": "stay fairly high",
    "structure_score": "stay fairly high",

    "melody_overall_acc": "drop",
    "melody_voicing_recall": "stay high-ish",
    "melody_voicing_false_alarm": "mixed",
    "melody_pitch_acc": "drop",
    "melody_chroma_acc": "drop some",
    "melody_contour_dtw": "stay higher",
    "melody_motif_jaccard": "mixed/drop",
    "melody_motif_recall": "mixed/drop",
}

def print_metric_block(title, keys, metrics_dict):
    print(f"\n{title}")
    print("-" * 112)
    print(f"{'Metric':<40} {'Median':>10} {'Std':>10} {'N':>6} {'Expected':>20}")
    print("-" * 112)

    for name in keys:
        med, std, n = summarize(metrics_dict[name])
        exp = EXPECTED.get(name, "")
        if n > 0:
            print(f"{name:<40} {med:>10.4f} {std:>10.4f} {n:>6} {exp:>20}")
        else:
            print(f"{name:<40} {'N/A':>10} {'N/A':>10} {0:>6} {exp:>20}")

print("\n" + "=" * 112)
print(f"VOCAL-ONLY PITCH SHIFT (+{VOCAL_PITCH_SHIFT_STEPS} st) — {len(all_results)} attempted pairs, {len(metrics['melody_overall_acc'])} valid")
print("=" * 112)

print_metric_block(
    "HARMONY (should stay comparatively more stable)",
    [
        "harmony_key_distance",
        "harmony_chroma_dtw",
        "harmony_chroma_mean",
        "harmony_best_shift_chroma_dtw",
        "harmony_best_shift_chroma_mean",
        "harmony_chord_root",
        "harmony_chord_majmin",
    ],
    metrics,
)

print_metric_block(
    "RHYTHM (should stay comparatively more stable)",
    [
        "rhythm_delta_bpm",
        "rhythm_delta_bpm_folded",
        "rhythm_beat_fmeasure",
        "rhythm_info_gain",
    ],
    metrics,
)

print_metric_block(
    "STRUCTURE (should stay comparatively more stable)",
    [
        "structure_boundary_f",
        "structure_ari",
        "structure_score",
    ],
    metrics,
)

print_metric_block(
    "MELODY (target facet: should change most)",
    [
        "melody_overall_acc",
        "melody_voicing_recall",
        "melody_voicing_false_alarm",
        "melody_pitch_acc",
        "melody_chroma_acc",
        "melody_contour_dtw",
        "melody_motif_jaccard",
        "melody_motif_recall",
    ],
    metrics,
)

print("\n" + "=" * 112)

valid_examples = [r for r in all_results if "error" not in r]
if PRINT_ONE_EXAMPLE and valid_examples:
    print(json.dumps(valid_examples[0], indent=2))


✓ Built MUSDB lookup with 144 tracks


Creating vocal-only pitch-shift audio (+5 st): 100%|██████████| 144/144 [02:27<00:00,  1.03s/it]


✓ Created 144 valid vocal-only pitch-shift pairs


Evaluating batches:   0%|          | 0/5 [00:00<?, ?it/s]/home/eric/MuseCPEval/lib/python3.11/site-packages/mir_eval/segment.py:120: UserWarning: Estimated intervals are empty.
  warnings.warn("Estimated intervals are empty.")
/home/eric/MuseCPEval/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/eric/MuseCPEval/lib/python3.11/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/eric/MuseCPEval/lib/python3.11/site-packages/mir_eval/segment.py:120: UserWarning: Estimated intervals are empty.
  warnings.warn("Estimated intervals are empty.")
/home/eric/MuseCPEval/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/eric/MuseCPEval/lib/python3.11/site-packages/numpy/core/_methods.py:129: Runtime

✓ batch 1 done (32/144)


...

✓ batch 3 done (96/144)


/home/eric/MuseCPEval/lib/python3.11/site-packages/mir_eval/beat.py:91: UserWarning: Reference beats are empty.
  warnings.warn("Reference beats are empty.")
/home/eric/MuseCPEval/lib/python3.11/site-packages/mir_eval/beat.py:376: UserWarning: Only one estimated beat was provided, so beat intervals cannot be computed.
  warnings.warn("Only one estimated beat was provided, so beat intervals"
/home/eric/MuseCPEval/lib/python3.11/site-packages/mir_eval/beat.py:464: UserWarning: Only one estimated beat was provided, so beat intervals cannot be computed.
  warnings.warn("Only one estimated beat was provided, so beat intervals"
/home/eric/MuseCPEval/lib/python3.11/site-packages/mir_eval/beat.py:622: UserWarning: Only one estimated beat was provided, so beat intervals cannot be computed.
  warnings.warn("Only one estimated beat was provided, so beat intervals"
/home/eric/MuseCPEval/lib/python3.11/site-packages/mir_eval/segment.py:117: UserWarning: Reference intervals are empty.
  warnings.war

✓ batch 4 done (128/144)


/home/eric/MuseCPEval/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/eric/MuseCPEval/lib/python3.11/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/eric/MuseCPEval/lib/python3.11/site-packages/mir_eval/segment.py:117: UserWarning: Reference intervals are empty.
  warnings.warn("Reference intervals are empty.")
/home/eric/MuseCPEval/lib/python3.11/site-packages/mir_eval/segment.py:120: UserWarning: Estimated intervals are empty.
  warnings.warn("Estimated intervals are empty.")
/home/eric/MuseCPEval/lib/python3.11/site-packages/mir_eval/segment.py:117: UserWarning: Reference intervals are empty.
  warnings.warn("Reference intervals are empty.")
/home/eric/MuseCPEval/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(

✓ batch 5 done (144/144)
✓ Saved raw results to vocal_only_pitch_shift_results.json

VOCAL-ONLY PITCH SHIFT (+5 st) — 144 attempted pairs, 144 valid

HARMONY (should stay comparatively more stable)
----------------------------------------------------------------------------------------------------------------
Metric                                       Median        Std      N             Expected
----------------------------------------------------------------------------------------------------------------
harmony_key_distance                         0.0000     0.1048    144       small/moderate
harmony_chroma_dtw                           0.9895     0.0153    144     stay fairly high
harmony_chroma_mean                          0.9980     0.0063    144     stay fairly high
harmony_best_shift_chroma_dtw                0.9895     0.0137    144            stay high
harmony_best_shift_chroma_mean               0.9980     0.0048    144            stay high
harmony_chord_root            